### **Empezamos importando las librerías que vamos a usar para nuestro modelo de Clustering**

In [34]:
import csv
import re
import warnings
import json
import logging
from datetime import datetime
from pathlib import Path
from typing import List, Tuple, Optional, Dict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import seaborn as sns

# Para visualizaciones 3D interactivas
try:
    import plotly.express as px
    import plotly.graph_objects as go
    PLOTLY_AVAILABLE = True
except ImportError:
    PLOTLY_AVAILABLE = False
    PLOTLY_IMPORT_WARNING = "Plotly no disponible. Se generarán solo gráficos 2D."
else:
    PLOTLY_IMPORT_WARNING = None

# HDBSCAN
try:
    import hdbscan
except Exception as e:
    raise ImportError("Necesitas instalar 'hdbscan' (pip install hdbscan)") from e

from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.stats import anderson

from sklearn.preprocessing import StandardScaler, MaxAbsScaler, RobustScaler, LabelEncoder
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.model_selection import train_test_split
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score, adjusted_rand_score, pairwise_distances
from sklearn.neighbors import NearestNeighbors
from sklearn.exceptions import ConvergenceWarning
from sklearn.compose import ColumnTransformer

warnings.filterwarnings("ignore", category=FutureWarning)

### **Configuración general (Parámetros y carpetas a almacenar)**

In [35]:
# Archivo fuente y separador esperado (si SEP=None, se detecta automáticamente)
RAW_FILE = "afiliados_mas_usos.csv"
SEP = None #";"  # si no estás seguro, pon SEP=None para autodetección

# Carpetas de salida (mantenemos estructura)
OUTPUT_DIR = Path("./outputs") # Convert OUTPUT_DIR to a Path object
DIR_CLEAN = OUTPUT_DIR / "01_clean"
DIR_ANALYSIS = OUTPUT_DIR / "02_analysis"
DIR_MODELS = OUTPUT_DIR / "03_Resultados"  # reutilizamos esta carpeta para resultados de modelos

# Semilla y rangos
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
K_RANGE = list(range(90, 140))        # rango de K a explorar
N_NEIGHBORS_SC = 10                   # k-NN para Spectral
MIN_SELECTED_FEATURES = 6             # min features finales
MAX_SELECTED_FEATURES = 10            # max features finales

# Muestreo: si None => 100% datos; si entero => se toma N aleatorio y SE ASUME población completa
SAMPLE_N = None #None  # EJ: 5000 para trabajar solo con 5000 filas como "población"

### **Estructura Conjunto de Datos de entrada**

In [36]:
FULL_ORDER = [
    "DocEmp","TipoDocEmp","NomEmp","CodclaseEconomicaEmp","CodSeccionEconomicaEmp","SeccionEconomicaEmp","NitSinDV",
    "Femenino","Masculino","Desconocido",
    "TarifaA","TarifaB","TarifaC",
    "Jovenes","AdultoJoven","Adulto","AdultoMayor",
    "TotalAfiliados","Aporte","TotalUsos",
    "UsosCultura","UsosEducacion","UsosSalud","UsosEmpleo",
    "PorcentajeFemenino","PorcentajeMasculino","PorcentajeDesconocido",
    "PorcentajeTarifaA","PorcentajeTarifaB","PorcentajeTarifaC",
    "PorcentajeJovenes","PorcentajeAdultoJoven","PorcentajeAdulto","PorcentajeAdultoMayor"
]

#Los siguientes tres bloques son aquellos cuyos valores sumados por bloque es igual a 1
#Por ejemplo, %Femenino + %Masculino + %Desconocido = 100% (Normalizado es 1)
BLOCKS = {
    "GENERO": ["PorcentajeFemenino","PorcentajeMasculino","PorcentajeDesconocido"],  # 3 -> ILR 2
    "TARIFA": ["PorcentajeTarifaA","PorcentajeTarifaB","PorcentajeTarifaC"],          # 3 -> ILR 2
    "EDAD":   ["PorcentajeJovenes","PorcentajeAdultoJoven","PorcentajeAdulto","PorcentajeAdultoMayor"]  # 4 -> ILR 3
}

#SCALE_VARS = ["TotalAfiliados", "Aporte", "TotalUsos"]

### **Configuración mensajes y guardado en archivo .log**

In [37]:
# Logging
logger = logging.getLogger("pipeline_clustering")

#Nos aseguramos de, que existan las carpetas a la que vamos a dirigir las salidas
def ensure_dirs() -> None:
    """Crea carpetas de salida si no existen."""
    for d in [OUTPUT_DIR, DIR_CLEAN, DIR_ANALYSIS, DIR_MODELS]:
        d.mkdir(parents=True, exist_ok=True)

#Queremos que Los mensajes se guarden en el archivo .log, pero que no se limite solo a esto
def setup_logging() -> Path:
    """Configura logging a archivo y consola."""
    ensure_dirs()
    log_file = OUTPUT_DIR / f"pipeline_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    fmt = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
    fh = logging.FileHandler(log_file, encoding='utf-8'); fh.setLevel(logging.INFO); fh.setFormatter(fmt)
    ch = logging.StreamHandler(); ch.setLevel(logging.INFO); ch.setFormatter(fmt)
    logger.addHandler(fh); logger.addHandler(ch); logger.propagate = False
    return log_file

def log_info(msg: str) -> None:
    """Log nivel INFO a archivo y consola."""
    logger.info(msg)

def log_warning(msg: str) -> None:
    """Log nivel WARNING a archivo y consola."""
    logger.warning(msg)

### **Limpieza de Datos**

El archivo de lectura ".csv" necesita inicialmente ser limpiado, ya que presenta muchos caracteres innecesarios, falta el tipeo de las variables y no se puede trabajar con estos datos fácilmente sin tratamiento previo.

In [38]:
# LECTURA Y LIMPIEZA
def detect_csv_separator(file_path: str) -> str:
    """Detecta separador (',' o ';') inspeccionando las primeras líneas; fallback ';'."""
    log_info(f"Detectando separador para: {file_path}")
    separators = [',', ';']
    try:
        with open(file_path, 'r', encoding='utf-8-sig') as f:
            lines = [ln.strip() for _, ln in zip(range(5), f) if ln.strip()]
        best_sep, best_score = ';', -1
        for sep in separators:
            counts, ok = [], True
            for line in lines:
                try:
                    reader = csv.reader([line], delimiter=sep, quotechar='"')
                    fields = next(reader); counts.append(len(fields))
                except Exception:
                    ok = False; break
            if not ok or not counts:
                continue
            score = (100 if len(set(counts)) == 1 else 0) + np.mean(counts)
            if 30 <= np.mean(counts) <= 40: score += 50
            if score > best_score: best_score, best_sep = score, sep
        log_info(f"Separador detectado: '{best_sep}'")
        return best_sep
    except Exception as e:
        log_warning(f"No se pudo detectar separador ({e}). Uso ';'")
        return ';'

def clean_csv_file(input_file: str, output_file: str) -> None:
    """
    Normaliza el CSV de entrada:
    - Detecta separador
    - Asegura headers
    - Limpia espacios extra
    - Escribe CSV limpio con ';'
    - Elimina BOM si está presente en el header
    """
    expected_headers = FULL_ORDER
    sep = detect_csv_separator(input_file) if SEP is None else SEP

    rows = []
    has_headers = False
    try:
        with open(input_file, 'r', encoding='utf-8-sig') as f:
            reader = csv.reader(f, delimiter=sep, quotechar='"')
            rows = list(reader)

        if not rows:
            log_warning(f"Archivo de entrada vacío: {input_file}")
            return

        if rows[0] and rows[0][0].startswith('\ufeff'):
            rows[0][0] = rows[0][0][1:]
            log_info("BOM character removed from the first header.")

        clean_first_row = [c.strip() for c in rows[0]]
        matches = sum(1 for h in expected_headers if h in clean_first_row)
        has_headers = matches >= int(0.7 * len(expected_headers))
        log_info(f"Headers detectados: {has_headers} (coincidencias={matches})")

    except Exception as e:
        log_warning(f"Error processing input file for cleaning ({e}).")
        return


    with open(output_file, 'w', newline='', encoding='utf-8') as outfile:
        writer = csv.writer(outfile, delimiter=';', quotechar='"', quoting=csv.QUOTE_MINIMAL)

        if not has_headers:
            writer.writerow(expected_headers)
            data_rows = rows
        else:
            writer.writerow(clean_first_row)
            data_rows = rows[1:]

        for i, row in enumerate(data_rows, start=1 if not has_headers else 2):
            cleaned = []
            for field in row:
                field = re.sub(r"\s+", " ", field).strip()
                cleaned.append(field)
            if len(cleaned) > len(expected_headers):
                cleaned = cleaned[:len(expected_headers)]
                log_warning(f"Línea {i}: demasiadas columnas -> truncada.")
            while len(cleaned) < len(expected_headers): cleaned.append("")
            writer.writerow(cleaned)

    log_info(f"CSV limpio → {output_file}")


def cargar_y_tipear(path_csv: str, sep: str) -> pd.DataFrame:
    """Lee CSV y convierte columnas numéricas conocidas a float; el resto queda como string."""
    df = pd.read_csv(path_csv, sep=sep, dtype=str)
    log_info(f"Columnas del DataFrame después de cargar y tipear: {df.columns.tolist()}")
    numeric_cols = [
        "Femenino","Masculino","Desconocido",
        "TarifaA","TarifaB","TarifaC",
        "Jovenes","AdultoJoven","Adulto","AdultoMayor",
        "TotalAfiliados","Aporte","TotalUsos",
        "UsosCultura","UsosEducacion","UsosSalud","UsosEmpleo",
        "PorcentajeFemenino","PorcentajeMasculino","PorcentajeDesconocido",
        "PorcentajeTarifaA","PorcentajeTarifaB","PorcentajeTarifaC",
        "PorcentajeJovenes","PorcentajeAdultoJoven","PorcentajeAdulto","PorcentajeAdultoMayor"
    ]
    for c in [c for c in numeric_cols if c in df.columns]:
        s = df[c].astype(str).str.replace(",", ".", regex=False).str.replace(" ", "", regex=False)
        df[c] = pd.to_numeric(s, errors="coerce")
    return df

### **Asegurar índices - conservarlos al transformar**

In [39]:
def ensure_row_id(df: pd.DataFrame, id_col: str = "RowID", as_index: bool = True) -> pd.DataFrame:
    """
    Crea (si no existe) una columna RowID con un identificador estable de fila.
    Opcionalmente lo deja como índice para que todas las transformaciones preserven el alineamiento.
    """
    if id_col not in df.columns:
        df = df.copy()
        df[id_col] = np.arange(len(df), dtype=np.int64)
    if as_index:
        if df.index.name != id_col:
            df = df.set_index(id_col, drop=False)
    return df


### **Usamos ILR para trabajar con los bloques de columnas cuyas entradas suman 1 (100%), es decir, las de porcentaje**

In [40]:
def closure(x: np.ndarray) -> np.ndarray:
    """Cierre composicional: normaliza a suma 1; devuelve NaNs si suma <= 0, hay negativos o no finitos."""
    x = np.asarray(x, dtype=float)
    if np.any(~np.isfinite(x)) or np.any(x < 0):
        return np.full_like(x, np.nan, dtype=float)
    s = np.nansum(x)
    if s <= 0:
        return np.full_like(x, np.nan, dtype=float)
    return x / s


def multiplicative_zero_replacement(row: np.ndarray, tiny: float = 1e-6) -> np.ndarray:
    """
    Reemplazo multiplicativo de ceros (simple y estable):
    - Requiere vector no negativo con suma > 0.
    - Reemplaza ceros por 'add' y reduce proporcionalmente las partes no cero para conservar la suma original.
    - Devuelve una composición estrictamente positiva (no normalizada).
      (El cierre posterior se encarga de normalizar a suma 1).
    """
    x = np.asarray(row, dtype=float).copy()

    # Validaciones básicas
    if np.any(~np.isfinite(x)) or np.any(x < 0):
        return np.full_like(x, np.nan, dtype=float)

    S = x.sum()
    if S <= 0:
        return np.full_like(x, np.nan, dtype=float)

    z = (x == 0)
    m = int(z.sum())
    if m == 0:
        # No hay ceros; retornar tal cual (el cierre lo normaliza después)
        return x

    # Cantidad mínima a asignar a cada cero (proporcional al total para estabilidad)
    add = max(tiny * S, tiny)
    # Evita que el ajuste exceda la masa disponible
    if m * add >= S:
        # Si el total es demasiado pequeño vs. m*add, retorna NaN para filtrar aguas arriba
        return np.full_like(x, np.nan, dtype=float)

    # Reemplazo de ceros
    x[z] = add
    # Ajuste proporcional de las partes no-cero para mantener la suma original S
    sum_nonzero = x[~z].sum()
    factor = (S - m * add) / sum_nonzero if sum_nonzero > 0 else np.nan
    if not np.isfinite(factor) or factor <= 0:
        return np.full_like(x, np.nan, dtype=float)

    x[~z] *= factor
    return x  # SIN normalizar; closure() hará la normalización


def ilr_pivot(x: np.ndarray) -> np.ndarray:
    """
    ILR (pivot coordinates):
    y_j = sqrt(j/(j+1)) * ln( g(x1..xj) / x_{j+1} ), j = 1..D-1
    Requiere x > 0 y finito.
    """
    x = np.asarray(x, dtype=float)
    if np.any(~np.isfinite(x)) or np.any(x <= 0):
        return np.full(max(len(x) - 1, 0), np.nan, dtype=float)

    D = len(x)
    if D < 2:
        return np.array([], dtype=float)

    logx = np.log(x)
    y = np.empty(D - 1, dtype=float)
    for j in range(1, D):
        gm = np.exp(logx[:j].mean())  # media geométrica de las primeras j partes
        y[j - 1] = np.sqrt(j / (j + 1.0)) * (np.log(gm) - logx[j])
    return y


def compute_block_ilr(df: pd.DataFrame, block_cols: List[str], block_name: str) -> pd.DataFrame:
    """
    Para un bloque composicional (porcentajes/partes no negativas):
      1) Reemplazo multiplicativo de ceros
      2) Cierre (closure)
      3) ILR (ilr_pivot)
    Devuelve DataFrame con columnas {block_name}_ILR_1..{D-1}.
    """
    log_info(f"Aplicando transformaciones composicionales e ILR para el bloque: {block_name}")

    cols = [c for c in block_cols if c in df.columns]
    if not cols:
        log_warning(f"No se encontraron columnas del bloque '{block_name}' en el DataFrame.")
        return pd.DataFrame(index=df.index)

    # Forzar a numérico (valores no convertibles -> NaN)
    block = df[cols].apply(pd.to_numeric, errors="coerce")

    def apply_compositional_ilr(row: pd.Series):
        vals = row.values.astype(float)

        # Validaciones rápidas por fila
        if np.any(~np.isfinite(vals)) or np.any(vals < 0) or np.nansum(vals) <= 0:
            d = max(len(cols) - 1, 0)
            return np.array([np.nan] * d, dtype=float)

        # 1) reemplazo de ceros (mantiene suma original)
        rz = multiplicative_zero_replacement(vals)
        if np.any(~np.isfinite(rz)):
            d = max(len(cols) - 1, 0)
            return np.array([np.nan] * d, dtype=float)

        # 2) cierre (normaliza a suma 1)
        rc = closure(rz)

        # 3) ILR
        return ilr_pivot(rc)

    ilr_data = block.apply(apply_compositional_ilr, axis=1, result_type="expand")

    # Asignar nombres de columnas y conservar índice
    if ilr_data is None or ilr_data.shape[1] == 0:
        log_warning(f"Bloque '{block_name}' no produjo componentes ILR (¿menos de 2 columnas válidas?).")
        return pd.DataFrame(index=df.index)

    ilr_col_names = [f"{block_name}_ILR_{i+1}" for i in range(ilr_data.shape[1])]
    ilr_data.columns = ilr_col_names
    ilr_data.index = df.index
    log_info(f"Columnas ILR generadas para el bloque '{block_name}': {ilr_col_names}")

    return ilr_data

### **Transformar con Log1p (Log[1+x]) a las columnas TotalAfiliados y Aportes**

Las variables de TotalAfiliados y Aportes suelen ser muy asimétricas, debido a que hay muchas empresas pequeñas, pero hay pocas que demasiado grandes. Luego, la dispersión de estas columnas vista desde una escala lineal es alta, en cambio, si usamos la escala logarítmica con Log(1+x), se pueden tratar cómodamente. Además, esto armoniza con los bloques ILR que también son de caracter logarítmico.

>#### **Nota Técnica: Uso de Log1p vs ILR en nuevas columnas**
>
> Si en el futuro se desea incluir las columnas de usos específicos (`UsosCultura`, `UsosEducacion`, etc.), se debe tener en cuenta lo siguiente:
> **Transformación Log1p** Es ideal si lo que interesa es la **intensidad** o magnitud absoluta del uso (ej. cuántos usos hubo). Se recomienda aplicar `np.log1p(Valor / TotalAfiliados)` para normalizar por tamaño de empresa, similar a `Log1p_IntensidadUsos`.
>
> **¿Son compatibles Log1p y ILR?** Sí. El algoritmo de clustering (HDBSCAN/FAMD) recibe variables numéricas transformadas. Log1p comprime escalas logarítmicas de magnitud, e ILR abre composiciones cerradas. Ambas producen variables en escalas "lineales" aptas para el modelo. Lo importante es no mezclar *conceptos* (no usar raw counts con porcentajes raw).

In [41]:
def apply_log1p_transformations(df: pd.DataFrame) -> pd.DataFrame:
    """
    Crea columnas:
      - Log1p_TotalAfiliados
      - Log1p_Aporte
      - Log1p_IntensidadUsos = log1p(TotalUsos / max(TotalAfiliados, 1))

    No modifica las columnas originales y conserva el índice tal cual.
    """
    log_info("Aplicando transformaciones log1p a variables seleccionadas.")

    required = ["TotalAfiliados", "Aporte", "TotalUsos"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        log_warning(f"No se aplican transformaciones: faltan columnas requeridas: {missing}")
        # devolvemos copia con el MISMO índice para no romper el pipeline
        out = df.copy()
        out.index = df.index
        return out

    out = df.copy()  # conserva índice y columnas

    # Parseo robusto a numérico sin tocar originales (usamos series temporales)
    ta = pd.to_numeric(out["TotalAfiliados"], errors="coerce")
    ap = pd.to_numeric(out["Aporte"],          errors="coerce")
    tu = pd.to_numeric(out["TotalUsos"],       errors="coerce")

    # Stats previas
    n = len(out)
    bad_ta = int((~np.isfinite(ta)).sum())
    bad_ap = int((~np.isfinite(ap)).sum())
    bad_tu = int((~np.isfinite(tu)).sum())
    if bad_ta or bad_ap or bad_tu:
        log_warning(f"Valores no finitos antes de log1p → "
                    f"TotalAfiliados:{bad_ta}/{n}, Aporte:{bad_ap}/{n}, TotalUsos:{bad_tu}/{n}")

    # “Sanitizar” para log1p (clip a >=0 y NaN→0) SIN tocar original
    ta_safe = ta.replace([np.inf, -np.inf], np.nan).fillna(0).clip(lower=0)
    ap_safe = ap.replace([np.inf, -np.inf], np.nan).fillna(0).clip(lower=0)
    tu_safe = tu.replace([np.inf, -np.inf], np.nan).fillna(0).clip(lower=0)

    # Nuevas columnas (índice preservado)
    out.loc[:, "Log1p_TotalAfiliados"] = np.log1p(ta_safe)
    out.loc[:, "Log1p_Aporte"] = np.log1p(ap_safe)
    intensidad = tu_safe / np.maximum(ta_safe, 1.0)  # evita división por cero
    out.loc[:, "Log1p_IntensidadUsos"] = np.log1p(intensidad)

    # Comprobación post
    for col in ["Log1p_TotalAfiliados", "Log1p_Aporte", "Log1p_IntensidadUsos"]:
        nf = (~np.isfinite(out[col])).sum()
        if nf:
            log_warning(f"La columna '{col}' contiene {int(nf)} valores no finitos tras log1p.")

    # Limpieza opcional de columna antigua
    if "AportePC" in out.columns:
        out = out.drop(columns=["AportePC"])

    # Garantiza, por si acaso, que el índice quede idéntico
    out.index = df.index

    log_info("Transformaciones log1p aplicadas (índice preservado).")
    return out

### **Dataframe que se utilizará para la clusterización**

In [42]:
def preparar_df_cluster(df: pd.DataFrame) -> pd.DataFrame:
    """
    Prepara el DataFrame para FAMD/clusterización seleccionando columnas ILR, Log1p y la categórica original.
    - Conserva el índice tal cual (RowID ya asegurado antes).
    - Asegura dtype category en 'SeccionEconomicaEmp' y rellena nulos con 'DESCONOCIDO'.

    Returns:
        DataFrame listo para FAMD con el MISMO índice que 'df'.
    """
    # 1) Candidatas
    ilr_prefixes = tuple(f"{block}_ILR_" for block in BLOCKS.keys())
    ilr_cols  = [c for c in df.columns if c.startswith(ilr_prefixes)]
    log1p_cols = [c for c in df.columns if c.startswith("Log1p_")]
    cat_col = "SeccionEconomicaEmp"

    # 2) Asegurar categórica
    if cat_col in df.columns:
        if df[cat_col].isna().any():
            # Etiqueta explícita para nulos: evita que FAMD los descarte silenciosamente
            df = df.copy()
            df[cat_col] = df[cat_col].fillna("DESCONOCIDO")
        if not isinstance(df[cat_col].dtype, pd.CategoricalDtype): # Updated line
            df = df.copy()
            df[cat_col] = df[cat_col].astype("category")
    else:
        log_warning("No se encontró la columna categórica 'SeccionEconomicaEmp'. FAMD funcionará sólo con numéricas.")

    # 3) Columnas efectivas (si existe la categórica, la ponemos al final)
    cols = ilr_cols + log1p_cols + ([cat_col] if cat_col in df.columns else [])
    cols = [c for c in cols if c in df.columns]

    if not cols:
        log_warning("No hay columnas válidas (ILR/Log1p/SeccionEconomicaEmp) para preparar el DataFrame de clusterización.")
        return pd.DataFrame(index=df.index)

    # 4) Selección manteniendo el índice (RowID)
    out = df[cols].copy()

    # 5) Pequeño sanity check
    n_ilr, n_log1p = len(ilr_cols), len(log1p_cols)
    has_cat = cat_col in out.columns
    log_info(f"DF clustering preparado: {out.shape[0]} filas, {out.shape[1]} cols "
             f"(ILR={n_ilr}, Log1p={n_log1p}, Categórica={'sí' if has_cat else 'no'}). "
             f"Índice preservado (p.ej. RowID).")
    try:
        display(out.head())
    except Exception:
        pass

    return out

### **Resumen estadístico de las variables**

In [43]:
def resumen_numericas_en_columnas(df_num: pd.DataFrame) -> pd.DataFrame:
    """
    Calcula un resumen estadístico por columna (métricas en filas) incluyendo conteos y % faltantes.
    """
    log_info("Calculando resumen estadístico para columnas numéricas.")
    desc = df_num.describe(percentiles=[0.25, 0.5, 0.75]).rename(index={"25%":"p25","50%":"p50","75%":"p75"})
    missing = df_num.isna().sum().to_frame().T; missing.index = ["missing"]
    missing_pct = (df_num.isna().mean()*100).round(2).to_frame().T; missing_pct.index=["missing_pct"]
    zeros = (df_num==0).sum().to_frame().T; zeros.index=["zeros"]
    zeros_pct = ((df_num==0).mean()*100).round(2).to_frame().T; zeros_pct.index=["zeros_pct"]
    unique_vals = df_num.nunique().to_frame().T; unique_vals.index=["unique_vals"]
    orden=["count","mean","std","min","p25","p50","p75","max","missing","missing_pct","zeros","zeros_pct","unique_vals"]
    resumen_df = pd.concat([desc,missing,missing_pct,zeros,zeros_pct,unique_vals], axis=0).reindex(orden)
    log_info("Resumen estadístico calculado.")
    return resumen_df

def resumen_categoricas(df: pd.DataFrame, cat_cols: List[str]) -> pd.DataFrame:
    """
    Genera resumen estadístico de columnas categóricas con moda, frecuencias y valores únicos.
    """
    log_info(f"Calculando resumen estadístico para columnas categóricas: {cat_cols}")
    if not cat_cols:
        log_warning("No se especificaron columnas categóricas para resumir.")
        return pd.DataFrame()

    resumen_data = []
    for col in cat_cols:
        if col in df.columns:
            serie = df[col].dropna()
            total_registros_col = len(df[col]) # Use total count before dropping NaNs

            if total_registros_col > 0: # Check total records for percentage calculation basis
                moda = serie.mode().iloc[0] if not serie.mode().empty else "N/A"
                freq_moda = serie.value_counts().max() if not serie.value_counts().empty else 0 # Use max freq if mode might return multiple
                # Recalculate pct_moda based on total non-null entries, or total entries depending on interpretation
                # Let's use total non-null entries for freq_moda percentage
                pct_moda = round((freq_moda / len(serie)) * 100, 2) if len(serie) > 0 else 0

                resumen_data.append({
                    'columna': col,
                    'total_registros': total_registros_col,
                    'valores_unicos': df[col].nunique(),
                    'valores_faltantes': df[col].isna().sum(),
                    'pct_faltantes': round(df[col].isna().mean() * 100, 2),
                    'moda': moda,
                    'frecuencia_moda': freq_moda,
                    'pct_moda': pct_moda
                })
            else:
                 # Handle case where column exists but has no data
                 resumen_data.append({
                    'columna': col,
                    'total_registros': 0,
                    'valores_unicos': 0,
                    'valores_faltantes': 0,
                    'pct_faltantes': 0.0,
                    'moda': "N/A",
                    'frecuencia_moda': 0,
                    'pct_moda': 0.0
                })


    resumen_df = pd.DataFrame(resumen_data)
    log_info("Resumen estadístico de categóricas calculado.")
    return resumen_df


def save_full_summary_to_excel(
    df: pd.DataFrame, # Receive the full dataframe
    categorical_cols: List[str],
    output_dir: Path = DIR_ANALYSIS
) -> None:
    """
    Calcula y guarda el resumen estadístico de columnas numéricas y categóricas
    en un archivo Excel con dos hojas.

    Args:
        df: DataFrame de entrada completo.
        categorical_cols: Lista de nombres de columnas categóricas a resumir.
        output_dir: Directorio donde guardar el archivo de resumen.
    """
    log_info("Iniciando cálculo y guardado de resumen completo en Excel.")

    # Separate numeric and categorical columns internally
    numeric_cols_df = df.select_dtypes(include=np.number)
    # Ensure categorical_cols actually exist in the dataframe
    valid_categorical_cols = [col for col in categorical_cols if col in df.columns]


    # Calcular resumen numérico
    numeric_summary_df = resumen_numericas_en_columnas(numeric_cols_df)

    # Calcular resumen categórico
    categorical_summary_df = resumen_categoricas(df, valid_categorical_cols) # Pass the full df and the list of cat cols


    # Define the output file path
    summary_file_path = output_dir / "data_summary.xlsx"

    try:
        # Asegurar que el directorio de destino existe
        output_dir.mkdir(parents=True, exist_ok=True)

        # Guardar en archivo Excel con múltiples hojas
        with pd.ExcelWriter(summary_file_path) as writer:
            if not numeric_summary_df.empty:
                numeric_summary_df.to_excel(writer, sheet_name='Resumen Numericas', index=True)
                log_info("Resumen numérico escrito en hoja 'Resumen Numericas'.")
            else:
                log_warning("DataFrame de resumen numérico vacío. No se creó la hoja 'Resumen Numericas'.")

            if not categorical_summary_df.empty:
                 # Set 'columna' as index for better Excel readability if desired, or keep as column
                 # categorical_summary_df.set_index('columna', inplace=True)
                 categorical_summary_df.to_excel(writer, sheet_name='Resumen Categoricas', index=False) # Keep columna as a regular column
                 log_info("Resumen categórico escrito en hoja 'Resumen Categoricas'.")
            else:
                log_warning("DataFrame de resumen categórico vacío. No se creó la hoja 'Resumen Categoricas'.")


        log_info(f"Resumen completo guardado en: {summary_file_path}")

    except Exception as e:
        log_warning(f"Error al guardar el resumen completo en Excel {summary_file_path}: {e}")


# Example usage in the pipeline:
# categorical_columns_to_summarize = ["SeccionEconomicaEmp", "TipoDocEmp"] # Replace with actual categorical columns
# save_full_summary_to_excel(df, categorical_columns_to_summarize, DIR_ANALYSIS)

### **Muestra y normalización para clustering**

In [44]:
def scale_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Escala todas las columnas numéricas del DataFrame con RobustScaler.
    Conserva el índice original (alineado al subconjunto numérico).
    """
    log_info("Iniciando escalado con RobustScaler para todas las columnas numéricas.")

    if df.empty:
        log_warning("El DataFrame para escalado está vacío.")
        return pd.DataFrame(index=df.index)

    # Solo columnas numéricas
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
    if not numeric_cols:
        log_warning("No se encontraron columnas numéricas para escalar.")
        return pd.DataFrame(index=df.index)

    # Subconjunto numérico (conserva índice) y tipado estable
    df_numeric = df[numeric_cols].copy().astype(np.float64)

    # Reemplaza infinitos por NaN por seguridad
    if not np.isfinite(df_numeric.values).all():
        log_warning("Columnas numéricas contienen Inf antes de RobustScaler. Reemplazando Inf con NaN.")
        df_numeric = df_numeric.replace([np.inf, -np.inf], np.nan)

    # Escalado robusto (no altera el orden de filas)
    try:
        scaler_robust = RobustScaler()
        scaled_data = scaler_robust.fit_transform(df_numeric)
        # Usa el índice del subconjunto numérico por si acaso
        df_scaled = pd.DataFrame(scaled_data, columns=numeric_cols, index=df_numeric.index)
        log_info(f"Escalado completado con RobustScaler. DataFrame final: {df_scaled.shape[0]} filas, {df_scaled.shape[1]} columnas.")
        display(df_scaled.head())
        return df_scaled
    except Exception as e:
        log_warning(f"Error durante el escalado con RobustScaler: {e}")
        return pd.DataFrame(index=df.index)


def sample_data(df: pd.DataFrame, sample_n: Optional[int], random_state: int) -> pd.DataFrame:
    """
    Toma una muestra aleatoria del DataFrame si sample_n está definido.
    Conserva el índice (RowID) tal cual.
    """
    if sample_n is not None and sample_n < len(df):
        log_info(f"Tomando una muestra de {sample_n} filas.")
        df_sampled = df.sample(n=sample_n, random_state=random_state).copy()
        # asegurar que el nombre del índice (RowID) se preserve (cosmético pero claro)
        df_sampled.index.name = df.index.name
        log_info(f"DataFrame muestreado: {df_sampled.shape[0]} filas.")
    else:
        log_info("No se ha especificado un tamaño de muestra o el tamaño es mayor/igual al total de filas. Usando todos los datos.")
        df_sampled = df.copy()
        df_sampled.index.name = df.index.name
    return df_sampled


### **Guardar CSV con datos escalados**

In [45]:
def save_dataframe_to_csv(df: pd.DataFrame, path: Path, index: bool = True) -> None:
    """
    Guarda un DataFrame a CSV. Por defecto, guarda el índice (RowID) para no perder trazabilidad.
    """
    path.parent.mkdir(parents=True, exist_ok=True)
    # Si el índice no tiene nombre, se lo ponemos por claridad
    if df.index.name is None:
        df.index.name = "RowID"
    df.to_csv(path, index=index)
    log_info(f"CSV guardado en: {path} (index={index})")

# Example usage in the pipeline:
# scaled_output_path = DIR_CLEAN / "afiliados_mas_usos_scaled.csv"
# save_dataframe_to_csv(cluster_df_scaled, scaled_output_path)

### **Análisis visual pre-clusterización**

Antes de ponera a prubea los distintos algoritmos de clustering, vamos a analizar el comportamiento de las variables que finales que se usarán para entrenar/ajustar el modelo y sus respectivos parámetros. Por esa razón, es importante fijarse en el comportamiento de cada variable y su relación con las demás, asimismo, su significancia.

#### **Correlación entre variables**

In [46]:
def analyze_correlation(df: pd.DataFrame, output_dir: Path = DIR_ANALYSIS) -> None:
    """
    Realiza un análisis de correlación en el DataFrame de entrada (esperando
    columnas numéricas como ILR y Log1p), excluyendo columnas de codificación
    one-hot de sección económica, y genera pairplot y heatmap.

    Nota: Esta función calcula la correlación en los datos numéricos proporcionados,
    independientemente de si están escalados o no. Si se usa antes del escalado,
    la interpretación de los valores de correlación debe considerar la escala original
    de las variables.

    Args:
        df: DataFrame de entrada, que se espera contenga columnas numéricas
            (como las de ILR y Log1p).
        output_dir: Directorio donde guardar los gráficos de correlación.
    """
    log_info("Iniciando análisis de correlación.")

    if df.empty:
        log_warning("El DataFrame para análisis de correlación está vacío.")
        return

    # Seleccionar solo columnas numéricas que no son one-hot encoded de Seccion Economica
    # We are interested in correlating the ILR and Log1p variables specifically for pre-FAMD analysis
    ilr_cols = [c for c in df.columns if c.startswith(tuple(f"{block}_ILR_" for block in BLOCKS.keys()))]
    log1p_cols = [c for c in df.columns if c.startswith("Log1p_")]

    # Combine ILR and Log1p columns for correlation analysis
    correlation_vars = ilr_cols + log1p_cols

    if not correlation_vars:
        log_warning("No hay columnas ILR_ o Log1p_ en el DataFrame para el análisis de correlación.")
        return

    df_corr = df[correlation_vars].copy() # Work only with the selected numeric columns

    # Asegurar que no haya NaN/Inf antes de calcular la correlación
    if not np.isfinite(df_corr.values).all():
         log_warning("El DataFrame para correlación contiene NaN/Inf. Se omitirán las filas con valores no finitos para el cálculo.")
         df_corr = df_corr.dropna(how='any')
         if df_corr.empty:
             log_warning("DataFrame vacío después de eliminar filas con NaN/Inf. No se puede calcular la correlación.")
             return


    # Calcular matriz de correlación
    correlation_matrix = df_corr.corr()
    log_info("Matriz de correlación calculada.")

    # Asegurar que el directorio de destino existe
    output_dir.mkdir(parents=True, exist_ok=True)

    # 1. Generar y guardar Pairplot
    try:
        log_info("Generando Pairplot (puede tardar)...")
        # Limitar el número de variables para evitar gráficos ilegibles en Pairplot
        if len(correlation_vars) > 15: # Adjusted limit for potentially more Log1p/ILR variables
            log_warning(f"Demasiadas variables ({len(correlation_vars)}) para un Pairplot legible. Se omitirá el Pairplot.")
        else:
            # Adjusting Pairplot to use the filtered df_corr
            pairplot_fig = sns.pairplot(df_corr)
            pairplot_path = output_dir / "correlation_pairplot_pre_famd.png" # Added suffix for clarity
            pairplot_fig.savefig(pairplot_path)
            plt.close(pairplot_fig.fig) # Close the figure to free memory
            log_info(f"Pairplot guardado en: {pairplot_path}")
    except Exception as e:
        log_warning(f"Error al generar/guardar Pairplot: {e}")
        plt.close('all') # Ensure all figures are closed in case of error


    # 2. Generar y guardar Heatmap de correlación
    try:
        log_info("Generando Heatmap de correlación.")
        plt.figure(figsize=(12, 10))
        # Adjusting Heatmap to use the correlation_matrix calculated from df_corr
        heatmap_fig = sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
        plt.title('Heatmap de Correlación (ILR y Log1p variables, pre-FAMD)') # Adjusted title
        plt.tight_layout() # Adjust layout to prevent labels overlapping
        heatmap_path = output_dir / "correlation_heatmap_pre_famd.png" # Added suffix for clarity
        plt.savefig(heatmap_path)
        plt.close() # Close the figure
        log_info(f"Heatmap guardado en: {heatmap_path}")
    except Exception as e:
        log_warning(f"Error al generar/guardar Heatmap: {e}")
        plt.close('all') # Ensure all figures are closed in case of error


    log_info("Análisis de correlación completado.")

### **Reducción de dimensionalidad TSNE y FAMD**

In [47]:
def reduce_dimensionality_tsne(
    df_scaled: pd.DataFrame,
    n_components: int,
    random_state: int = RANDOM_STATE,
    perplexity: int = 30,
    n_iter: int = 1000
) -> pd.DataFrame:
    """
    Reduce la dimensionalidad del DataFrame escalado usando t-SNE.

    Args:
        df_scaled: DataFrame escalado (sin NaN/Inf).
        n_components: Número de dimensiones de salida (2 o 3 típicamente).
        random_state: Semilla para reproducibilidad.
        perplexity: Valor de perplexidad (relacionado con tamaño de muestra).
        n_iter: Número de iteraciones de optimización.

    Returns:
        DataFrame con los componentes t-SNE (TSNE1..TSNEn).
    """
    log_info(f"Iniciando reducción t-SNE a {n_components} componentes.")

    if df_scaled.empty:
        log_warning("El DataFrame para t-SNE está vacío.")
        return pd.DataFrame(index=df_scaled.index)

    n_samples = df_scaled.shape[0]
    if n_samples < 30:
        log_warning("Demasiadas pocas muestras para t-SNE (<30). Usa PCA o aumenta el dataset.")
        return pd.DataFrame(index=df_scaled.index)

    # Ajuste automático de perplexidad
    perplexity = min(perplexity, n_samples - 1)
    if perplexity <= 1:
        log_warning("Perplexidad inválida para t-SNE. Considera usar PCA o más datos.")
        return pd.DataFrame(index=df_scaled.index)

    reducer = TSNE(
        n_components=n_components,
        random_state=random_state,
        perplexity=perplexity,
        n_iter=n_iter
    )
    reduced_components = reducer.fit_transform(df_scaled)

    component_names = [f"TSNE{i+1}" for i in range(n_components)]
    reduced_df = pd.DataFrame(reduced_components, columns=component_names, index=df_scaled.index)

    log_info(f"Reducción t-SNE completada ({n_components} componentes).")
    display(reduced_df.head())
    return reduced_df

#### **Reducción con FAMD**

In [48]:
!pip install prince

In [49]:
try:
    import prince
except ImportError:
    raise ImportError("Necesitas instalar 'prince' (pip install prince)")

def reduce_dimensionality_famd(
    df: pd.DataFrame,
    categorical_col: str,
    n_components: int,
    random_state: int = RANDOM_STATE
) -> pd.DataFrame:
    """
    Reduce la dimensionalidad del DataFrame usando FAMD (Factor Analysis of Mixed Data).
    Calcula y registra la varianza explicada (información conservada) por los
    componentes seleccionados utilizando los autovalores.

    Asume que el DataFrame contiene columnas numéricas (ILR, Log1p) y una
    única columna categórica especificada.

    Args:
        df: DataFrame de entrada con columnas numéricas y categóricas.
        categorical_col: Nombre de la única columna categórica a usar.
        n_components: Número de componentes principales a retener.
        random_state: Semilla para reproducibilidad.

    Returns:
        DataFrame con los componentes FAMD.
    """
    log_info(f"Iniciando reducción FAMD a {n_components} componentes, incluyendo '{categorical_col}'.")

    if df.empty:
        log_warning("El DataFrame para FAMD está vacío.")
        return pd.DataFrame(index=df.index)

    # Identificar columnas numéricas (asumiendo que son las que no son categóricas)
    # Asegurarse de seleccionar solo columnas numéricas que NO sean la categórica
    numeric_cols = [col for col in df.select_dtypes(include=np.number).columns.tolist() if col != categorical_col]

    # Asegurar que la columna categórica especificada existe
    if categorical_col not in df.columns:
        log_warning(f"La columna categórica '{categorical_col}' no se encontró en el DataFrame. No se puede aplicar FAMD.")
        return pd.DataFrame(index=df.index)

    # Asegurar que la columna categórica sea tratada como categórica
    df[categorical_col] = df[categorical_col].astype('category')

    # Crear una lista de columnas a incluir en FAMD
    # Necesitamos las columnas numéricas y la única categórica
    famd_cols = numeric_cols + [categorical_col]

    # Filtrar solo las columnas que realmente existen en el DataFrame
    famd_cols = [c for c in famd_cols if c in df.columns]

    if not famd_cols:
        log_warning("No hay columnas válidas (numéricas o categórica especificada) para aplicar FAMD.")
        return pd.DataFrame(index=df.index)

    # --- Selección columnas para FAMD (conserva índice) ---
    df_famd = df[famd_cols].copy()

    # --- Limpieza segura antes de FAMD (preserva índice) ---
    # 1) Asegura tipo categórico sin romper índice
    df_famd[categorical_col] = df_famd[categorical_col].astype('category')

    # 2) Reemplaza Inf por NaN solo en numéricas
    numeric_cols = [col for col in df_famd.select_dtypes(include=np.number).columns if col != categorical_col]
    numeric_data_famd = df_famd[numeric_cols].replace([np.inf, -np.inf], np.nan)

    # 3) Calcula índices válidos:
    #    - filas SIN NaN en numéricas
    #    - filas con categórica no nula (si quieres permitir NaN en categórica, usa idx_ok_numeric solamente)
    idx_ok_numeric = numeric_data_famd.dropna(how='any').index
    idx_ok_categ  = df_famd[df_famd[categorical_col].notna()].index
    rows_to_use = idx_ok_numeric.intersection(idx_ok_categ)

    if rows_to_use.empty:
        log_warning("DataFrame vacío después de filtrar filas no válidas para FAMD.")
        return pd.DataFrame(index=df.index)

    # 4) Subconjunto limpio (MISMO índice, reducido)
    if len(rows_to_use) < len(df_famd):
        log_info(f"FAMD: se excluyen {len(df_famd) - len(rows_to_use)} filas por NaN/Inf; se procesan {len(rows_to_use)} filas.")

    df_famd = df_famd.loc[rows_to_use].copy()
    numeric_data_famd = df_famd[numeric_cols]  # actualizado
    # --- fin limpieza segura ---

    # Aplicar FAMD
    try:
        # prince.FAMD espera un DataFrame con tipos mixtos
        # Corrige el parámetro 'engine'. Las opciones válidas son 'fbpca', 'scipy' o 'sklearn'.
        # Se usa 'sklearn' como opción robusta por defecto.
        # Se ajusta el modelo con n_components igual al número de columnas para obtener todos los autovalores
        # Esto evita el problema de 'n_components=None' con el motor sklearn
        num_cols_famd = df_famd.shape[1]
        famd = prince.FAMD(
            n_components=num_cols_famd, # Ajuste con número de columnas para obtener todos los autovalores
            n_iter=3, # Iteraciones por defecto, ajustar si es necesario
            copy=True,
            check_input=True,
            engine='sklearn', # Corrected engine parameter
            random_state=random_state
        )

        # Ajustar el modelo a los datos
        famd.fit(df_famd)

        # Calcular y registrar la varianza explicada usando los autovalores
        explained_variance_ratio = None
        if hasattr(famd, 'eigenvalues_') and famd.eigenvalues_ is not None and famd.eigenvalues_.size > 0:
             eigenvalues = np.array(famd.eigenvalues_)
             total_eigenvalue_sum = np.sum(eigenvalues)

             if total_eigenvalue_sum > 1e-9: # Evitar división por un valor cercano a cero
                 # Suma de los autovalores para los primeros n_components (o menos si se ajustaron menos componentes)
                 num_components_fitted = len(famd.components_) if hasattr(famd, 'components_') else len(eigenvalues)
                 components_to_sum = min(n_components, num_components_fitted)

                 if components_to_sum > 0:
                     sum_of_first_n_eigenvalues = np.sum(eigenvalues[:components_to_sum])
                     explained_variance_ratio = sum_of_first_n_eigenvalues / total_eigenvalue_sum
                     log_info(f"Varianza explicada por los primeros {components_to_sum} componentes FAMD (calculada de autovalores): {explained_variance_ratio:.4f} ({explained_variance_ratio*100:.2f}%)")

                     if explained_variance_ratio * 100.0 >= 90.0:
                         log_info(f"¡La varianza explicada ({explained_variance_ratio*100:.2f}%) es mayor o igual al 90% requerido!")
                     else:
                         log_warning(f"La varianza explicada ({explained_variance_ratio*100:.2f}%) es menor al 90% requerido. Considera aumentar n_components.")
                 else:
                     log_warning(f"No se pudo calcular la varianza explicada para {n_components} componentes (cero o componentes ajustados insuficientes).")
             else:
                 log_warning("La suma total de autovalores es cercana a cero. No se pudo calcular la varianza explicada.")

        else:
             log_warning("No se pudieron obtener los autovalores del modelo FAMD. El atributo 'eigenvalues_' no está disponible o está vacío después de ajustar.")


        # Mostrar resumen de autovalores para un desglose detallado
        if hasattr(famd, 'eigenvalues_summary') and famd.eigenvalues_summary is not None:
             log_info("Resumen detallado de autovalores y varianza explicada por componente FAMD:")
             display(famd.eigenvalues_summary)
        else:
             log_warning("No se pudo obtener el resumen detallado de autovalores (eigenvalues_summary) del modelo FAMD.")


        # Ahora transformar los datos con el número *solicitado* de componentes
        # Se necesita una nueva instancia de FAMD configurada con los n_components deseados para la transformación
        famd_transform = prince.FAMD(
            n_components=n_components, # Usar el n_components solicitado para la transformación
            n_iter=3, # Iteraciones por defecto
            copy=True,
            check_input=True,
            engine='sklearn', # Motor corregido
            random_state=random_state
        )
        # Ajustar y transformar específicamente para la salida final
        # Nota: Este segundo ajuste es necesario porque prince.FAMD no permite cambiar n_components después del ajuste inicial
        # y el primer ajuste se realizó con n_components = num_cols_famd para obtener todos los autovalores.
        reduced_components = famd_transform.fit_transform(df_famd)


        # El resultado de fit_transform es un DataFrame con el índice del df_famd de entrada
        # El número de columnas debe coincidir con el número de componentes ajustados en la segunda instancia
        num_output_components = reduced_components.shape[1]
        component_names = [f"FAMD{i+1}" for i in range(num_output_components)]

        reduced_df = pd.DataFrame(reduced_components.values, columns=component_names, index=df_famd.index)


        log_info(f"Reducción FAMD completada ({reduced_df.shape[1]} componentes retenidos).")
        display(reduced_df.head())
        return reduced_df

    except Exception as e:
        log_warning(f"Error durante la reducción FAMD: {e}")
        return pd.DataFrame(index=df.index) # Return empty df in case of error

# Ejemplo de uso:
# Asumiendo que df_transformed contiene las columnas ILR, Log1p y SeccionEconomicaEmp
# reduced_df_famd = reduce_dimensionality_famd(df_transformed, "SeccionEconomicaEmp", n_components=10, random_state=RANDOM_STATE)
# display(reduced_df_famd.head())

In [50]:
def plot_famd_elbow(famd_model: prince.FAMD, output_dir: Path = DIR_ANALYSIS) -> None:
    """
    Genera un gráfico de codo para FAMD basado en la inercia acumulada explicada.

    Args:
        famd_model: Objeto prince.FAMD ajustado.
        output_dir: Directorio donde guardar el gráfico.
    """
    log_info("Generando gráfico de codo para FAMD.")

    if not hasattr(famd_model, 'explained_inertia_') or famd_model.explained_inertia_ is None:
        log_warning("El modelo FAMD no tiene el atributo 'explained_inertia_'. No se puede generar el gráfico de codo.")
        return

    inertia = np.array(famd_model.explained_inertia_)
    cumulative_inertia = np.cumsum(inertia)
    n_components = len(inertia)

    if n_components == 0:
        log_warning("No hay componentes en el modelo FAMD para graficar el codo.")
        return

    # Generar el gráfico
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, n_components + 1), cumulative_inertia, marker='o', linestyle='--')

    # Añadir etiquetas y título
    plt.xlabel("Número de Componentes")
    plt.ylabel("Inercia Acumulada Explicada")
    plt.title("Gráfico de Codo FAMD: Inercia Acumulada Explicada")
    plt.grid(True)

    # Opcional: Añadir una línea vertical heurística para el "codo" (requiere inspección visual)
    # Esto es solo una guía visual, no una determinación automática del óptimo.
    # Puedes ajustar este valor después de ver el gráfico.
    # Por ejemplo, si el codo parece estar alrededor de 8 componentes:
    # plt.axvline(x=8, color='r', linestyle=':', label='Posible Codo (Heurístico)')
    # plt.legend()


    # Asegurar que el directorio de destino existe
    output_dir.mkdir(parents=True, exist_ok=True)

    # Guardar el gráfico
    elbow_plot_path = output_dir / "famd_elbow_plot.png"
    plt.savefig(elbow_plot_path)
    plt.close() # Cerrar la figura para liberar memoria

    log_info(f"Gráfico de codo FAMD guardado en: {elbow_plot_path}")

### **Aplicando método de clusterización**

#### **Ajustar Hiper-Parámetros previo HDBSCAN**

In [51]:
!pip install optuna

In [52]:
import time
import json
import optuna

#def _eval_hdbscan_once(
#    X,
#    min_cluster_size: int,
#    min_samples: int,
#    cluster_selection_epsilon: float,
#    metric: str = "euclidean",
#    consider_noise_in_mean_prob: bool = False,
#    selection_method: str = "leaf"
#):
#    """Entrena HDBSCAN una vez y devuelve métricas clave."""
#    model = hdbscan.HDBSCAN(
#        min_cluster_size=min_cluster_size,
#        min_samples=min_samples,
#        cluster_selection_method=selection_method,  # 'leaf' da más granularidad
#        cluster_selection_epsilon=cluster_selection_epsilon,
#        metric=metric,
#        prediction_data=True
#    ).fit(X)
#
#    labels = model.labels_
#    probs  = model.probabilities_
#    mask   = (labels >= 0)
#
#    # mean_prob sobre puntos clusterizados (más informativo)
#    if consider_noise_in_mean_prob or not mask.any():
#        mean_prob = float(np.mean(probs))
#    else:
#        mean_prob = float(np.mean(probs[mask]))
#
#    # persistencia mediana
#    cp = getattr(model, "cluster_persistence_", None)
#    if cp is None:
#        median_persist = 0.0
#    else:
#        try:
#            arr = np.array(list(cp.values()))
#        except Exception:
#            arr = np.asarray(cp)
#        median_persist = float(np.median(arr)) if arr.size > 0 else 0.0
#
#    n_clusters = int(len(np.unique(labels[mask]))) if mask.any() else 0
#    noise_pct  = float((~mask).mean() * 100.0)
#
#    return mean_prob, median_persist, noise_pct, n_clusters, model

def _eval_hdbscan_once(
    X,
    min_cluster_size: int,
    min_samples: int,
    cluster_selection_epsilon: float,
    metric: str = "euclidean",
    consider_noise_in_mean_prob: bool = False,
    selection_method: str = "eom"
):
    """Entrena HDBSCAN una vez y devuelve métricas clave."""
    model = hdbscan.HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=min_samples,
        cluster_selection_method=selection_method,  # 'leaf' da más granularidad
        cluster_selection_epsilon=cluster_selection_epsilon,
        metric=metric,
        prediction_data=True
    ).fit(X)

    labels = model.labels_
    probs  = model.probabilities_
    mask   = (labels >= 0)

    # mean_prob sobre puntos clusterizados (más informativo)
    if consider_noise_in_mean_prob or not mask.any():
        mean_prob = float(np.mean(probs))
    else:
        mean_prob = float(np.mean(probs[mask]))

    # ===================== BLOQUE AJUSTADO (NUEVO) =====================
    # median_persist: mediana ponderada por tamaño de cluster,
    # pero SOLO sobre clusters "principales" (por tamaño relativo).
    cp = getattr(model, "cluster_persistence_", None)

    if cp is None or not mask.any():
        median_persist = 0.0
    else:
        labels_valid = labels[mask]  # sólo puntos asignados a clusters (sin ruido)
        cluster_ids, counts = np.unique(labels_valid, return_counts=True)

        total_points = counts.sum()
        if total_points == 0:
            median_persist = 0.0
        else:
            # fracción de puntos que aporta cada cluster
            size_frac = counts / float(total_points)

            # --- UMBRAL DE CLUSTERS PRINCIPALES ---
            # Ajusta este valor: 0.01 = clusters que tienen al menos el 1% de los puntos clusterizados
            MAIN_FRAC = 0.005

            main_mask = (size_frac >= MAIN_FRAC)
            main_cluster_ids = cluster_ids[main_mask]
            main_counts      = counts[main_mask]

            # si ningún cluster supera el umbral, usamos todos (fallback seguro)
            if main_cluster_ids.size == 0:
                main_cluster_ids = cluster_ids
                main_counts      = counts

            # convertimos persistencias a array para indexar por ID de cluster
            try:
                cp_arr = np.asarray(cp, dtype=float)
            except Exception:
                cp_arr = None

            pers_main = []
            sizes_main = []

            for cid, csize in zip(main_cluster_ids, main_counts):
                if isinstance(cp, dict):
                    p_c = float(cp.get(cid, 0.0))
                else:
                    if (cp_arr is not None) and (0 <= cid < cp_arr.size):
                        p_c = float(cp_arr[cid])
                    else:
                        p_c = 0.0

                pers_main.append(p_c)
                sizes_main.append(float(csize))

            pers_main  = np.asarray(pers_main, dtype=float)
            sizes_main = np.asarray(sizes_main, dtype=float)

            if pers_main.size == 0:
                median_persist = 0.0
            else:
                # mediana ponderada por tamaño de cluster
                order   = np.argsort(pers_main)
                p_sorted = pers_main[order]
                w_sorted = sizes_main[order]

                w_cum = np.cumsum(w_sorted) / w_sorted.sum()
                idx_m = np.searchsorted(w_cum, 0.5)
                median_persist = float(p_sorted[idx_m])
    # ================== FIN DEL BLOQUE AJUSTADO (NUEVO) =====================

    n_clusters = int(len(np.unique(labels[mask]))) if mask.any() else 0
    noise_pct  = float((~mask).mean() * 100.0)

    return mean_prob, median_persist, noise_pct, n_clusters, model


def tune_hdbscan_optuna(
    X,
    n_trials: int = 120,
    study_name: str = "hdbscan_tpe_comfama",
    storage: Optional[str] = None,
    save_best_to: Optional[str] = None,
    consider_noise_in_mean_prob: bool = False,
    # metas deseadas
    target_min_mean_prob: float = 0.80,
    target_min_persist: float = 0.30,
    target_max_noise: float = 20.0,
    target_min_clusters: int = 10,
    target_max_clusters: int = 50,
):
    """
    Tunea HDBSCAN con TPE maximizando un score compuesto:
      + mean_prob alto (ideal ≥ target_min_mean_prob)
      + persistencia mediana ≥ target_min_persist
      + ruido <= target_max_noise
      + n_clusters dentro de [target_min_clusters, target_max_clusters]

    Devuelve dict con hiperparámetros óptimos.
    """
    N = X.shape[0]

    if storage:
        log_info(f"[Optuna] Usando storage: {storage}")
        study = optuna.create_study(
            direction="maximize",
            study_name=study_name,
            storage=storage,
            load_if_exists=True
        )
    else:
        study = optuna.create_study(
            direction="maximize",
            study_name=study_name,
        )

    log_info(f"[Optuna] Study name: {study.study_name}")

    def objective(trial: optuna.Trial):
        # --- espacio de búsqueda de hiperparámetros ---

        # min_cluster_size proporcional a N
        frac_mcs = trial.suggest_float("frac_min_cluster_size", 0.0003, 0.018, log=True)  # 0.05%–1%
        min_cluster_size = max(10, int(frac_mcs * N))   # subo un poco el mínimo

        # min_samples como proporción de min_cluster_size
        # garantizamos min_samples <= min_cluster_size
        ms_ratio = trial.suggest_float("min_samples_ratio", 0.1, 0.8)
        min_samples = max(5, int(ms_ratio * min_cluster_size))

        # epsilon (incluye valores más grandes para permitir muchos clusters,
        # que ahora podrán ser penalizados si se pasan de target_max_clusters)
        cluster_selection_epsilon = trial.suggest_float(
            "cluster_selection_epsilon",
            0.0,
            0.4
        )

        # métrica
        metric = trial.suggest_categorical("metric", ["euclidean", "manhattan"])

        # --- eval HDBSCAN ---
        mean_prob, median_persist, noise_pct, n_clusters, _ = _eval_hdbscan_once(
            X,
            min_cluster_size=min_cluster_size,
            min_samples=min_samples,
            cluster_selection_epsilon=cluster_selection_epsilon,
            metric=metric,
            consider_noise_in_mean_prob=consider_noise_in_mean_prob,
            selection_method="eom"
        )

        # --- score compuesto (prioridad fuerte en median_persist) ---

        score = 0.0

        # 1) median_persist: protagonista del score
        score += 1.0 * median_persist

        # 2) penalización si mean_prob está por debajo del umbral
        if mean_prob < target_min_mean_prob:
            deficit = target_min_mean_prob - mean_prob
            score -= 1.5 * deficit  # sigue siendo importante, pero ya no domina

        # pequeño bonus si mean_prob es claramente buena
        if mean_prob >= target_min_mean_prob:
            score += 0.1 * min(
                1.0,
                (mean_prob - target_min_mean_prob) / (1.0 - target_min_mean_prob + 1e-9)
            )

        # 3) ruido: penaliza sólo por encima de target_max_noise
        if noise_pct > target_max_noise:
            score -= 0.02 * (noise_pct - target_max_noise)

        # 4) número de clusters: penaliza estar fuera del rango deseado,
        #    pero con coeficientes más suaves que antes
        if n_clusters < target_min_clusters:
            score -= 0.01 * (target_min_clusters - n_clusters)
        elif n_clusters > target_max_clusters:
            score -= 0.02 * (n_clusters - target_max_clusters)

        # --- logging bonito por trial ---
        log_info(
            f"[Optuna-trial] trial={trial.number} | "
            f"min_cluster_size={min_cluster_size}, min_samples={min_samples}, "
            f"eps={cluster_selection_epsilon:.3f}, metric={metric} || "
            f"mean_prob={mean_prob:.3f}, median_persist={median_persist:.3f}, "
            f"noise={noise_pct:.1f}%, n_clusters={n_clusters}, score={score:.3f}"
        )

        # --- guardar métricas en user_attrs ---
        trial.set_user_attr("mean_prob", mean_prob)
        trial.set_user_attr("median_persist", median_persist)
        trial.set_user_attr("noise_pct", noise_pct)
        trial.set_user_attr("n_clusters", n_clusters)
        trial.set_user_attr("min_cluster_size", min_cluster_size)
        trial.set_user_attr("min_samples", min_samples)
        trial.set_user_attr("metric", metric)
        trial.set_user_attr("cluster_selection_epsilon", cluster_selection_epsilon)
        trial.set_user_attr("cluster_selection_method", "eom")

        return float(score)

    t0 = time.time()
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    elapsed = time.time() - t0

    t = study.best_trial
    best = {
        "min_cluster_size": int(t.user_attrs["min_cluster_size"]),
        "min_samples": int(t.user_attrs["min_samples"]),
        "cluster_selection_epsilon": float(t.user_attrs["cluster_selection_epsilon"]),
        "metric": t.user_attrs["metric"],
        "mean_prob": float(t.user_attrs["mean_prob"]),
        "median_persist": float(t.user_attrs["median_persist"]),
        "noise_pct": float(t.user_attrs["noise_pct"]),
        "n_clusters": int(t.user_attrs["n_clusters"]),
        "optuna_value": float(t.value),
        "trials": len(study.trials),
        "elapsed_sec": round(elapsed, 2),
        "cluster_selection_method": "eom",
    }

    log_info(f"[Optuna] Mejor combinación: {best}")

    if save_best_to:
        import pathlib
        p = pathlib.Path(save_best_to)
        p.parent.mkdir(parents=True, exist_ok=True)
        with p.open("w", encoding="utf-8") as f:
            json.dump(best, f, ensure_ascii=False, indent=2)
        log_info(f"[Optuna] Parámetros guardados en: {p}")

    return best, study

#### **Método HDBSCAN**

In [53]:
def apply_hdbscan_clustering(
    df_scaled: pd.DataFrame,
    min_cluster_size: int = 270,
    min_samples: int = 15, #Optional[int] = None,
    metric: str = "manhattan",#"euclidean",
    cluster_selection_method: str = "leaf",
    cluster_selection_epsilon: float = 0.0, # Added parameter
    allow_single_cluster: bool = False,
) -> Tuple[np.ndarray, Optional[hdbscan.HDBSCAN], Dict[str, Optional[float]]]:
    """
    Aplica HDBSCAN y calcula métricas de desempeño.

    Args:
        df_scaled: DataFrame/matriz con datos escalados y sin NaN/Inf.
        min_cluster_size: Tamaño mínimo de cluster (controla resolución).
        min_samples: Puntos mínimos para considerar núcleo (robustez al ruido). Si None, usa min_cluster_size.
        metric: Métrica de distancia (euclidean, manhattan, cosine, etc.).
        cluster_selection_method: 'eom' (default) o 'leaf'.
        cluster_selection_epsilon: Distancia para fusionar clusters (solo para cluster_selection_method='leaf'). # Updated description
        allow_single_cluster: Permite 1 solo cluster si la estructura lo sugiere.

    Returns:
        (labels, model, metrics)
            - labels: np.ndarray de etiquetas (-1 = ruido).
            - model: objeto hdbscan.HDBSCAN ajustado o None si falla.
            - metrics: dict con:
                * n_clusters: número de clusters (excluye ruido)
                * noise_ratio: proporción de ruido
                * silhouette: silhouette score (solo si >=2 clusters y >= 2 muestras por cluster)
                * calinski_harabasz: Calinski–Harabasz (solo si >=2 clusters)
                * davies_bouldin: Davies–Bouldin (solo si >=2 clusters)
                * mean_probability: probabilidad media de pertenencia (solo puntos asignados)
                * median_persistence_raw: mediana simple por cluster
                * median_persistence_main_weighted: mediana ponderada por tamaño usando solo clusters “principales” (MAIN_FRAC)
    """
    # Preparación y validaciones
    X = df_scaled.values if isinstance(df_scaled, pd.DataFrame) else np.asarray(df_scaled, dtype=float)
    metrics: Dict[str, Optional[float]] = {
    "n_clusters": None,
    "noise_ratio": None,
    "silhouette": None,
    "calinski_harabasz": None,
    "davies_bouldin": None,
    "mean_probability": None,
    "median_persistence_raw": None,
    "median_persistence_main_weighted": None,
    }

    if X.size == 0:
        log_warning("apply_hdbscan_clustering: dataset vacío.")
        return np.array([]), None, metrics
    if not np.isfinite(X).all():
        log_warning("apply_hdbscan_clustering: hay NaN/Inf; limpia antes de clusterizar.")
        return np.array([]), None, metrics
    if X.shape[0] < max(5, min_cluster_size):
        log_warning(f"apply_hdbscan_clustering: pocas filas ({X.shape[0]}); sube datos o baja min_cluster_size.")
        return np.array([]), None, metrics

    # Ajuste del modelo
    log_info(
        f"HDBSCAN(metric={metric}, min_cluster_size={min_cluster_size}, "
        f"min_samples={min_samples if min_samples is not None else min_cluster_size}, "
        f"cluster_selection_method={cluster_selection_method}, "
        f"cluster_selection_epsilon={cluster_selection_epsilon}, " # Included in log
        f"allow_single_cluster={allow_single_cluster})"
    )

    model = hdbscan.HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=min_samples if min_samples is not None else min_cluster_size,
        metric=metric,
        cluster_selection_method=cluster_selection_method,
        cluster_selection_epsilon=cluster_selection_epsilon, # Pass the parameter
        allow_single_cluster=allow_single_cluster,
        prediction_data=True,  # para probabilities_ y soft clustering
    ).fit(X)

    labels = model.labels_
    unique_labels = np.unique(labels)
    n_clusters = int((unique_labels[unique_labels != -1]).size)
    noise_ratio = float(np.mean(labels == -1))

    metrics["n_clusters"] = n_clusters
    metrics["noise_ratio"] = noise_ratio

    log_info(f"HDBSCAN: clusters={n_clusters} (sin contar ruido), ruido={noise_ratio:.2%}")

    # Métricas solo sobre puntos asignados (excluye ruido)
    mask = labels != -1
    X_assigned = X[mask]
    y_assigned = labels[mask]
    assigned_cluster_count = np.unique(y_assigned).size

    if X_assigned.shape[0] >= 2 and assigned_cluster_count >= 2:
        try:
            metrics["silhouette"] = float(silhouette_score(X_assigned, y_assigned, metric="euclidean"))
        except Exception as e:
            log_warning(f"Silhouette no disponible: {e}")
        try:
            metrics["calinski_harabasz"] = float(calinski_harabasz_score(X_assigned, y_assigned))
        except Exception as e:
            log_warning(f"Calinski–Harabasz no disponible: {e}")
        try:
            metrics["davies_bouldin"] = float(davies_bouldin_score(X_assigned, y_assigned))
        except Exception as e:
            log_warning(f"Davies–Bouldin no disponible: {e}")
    else:
        log_warning("No hay suficientes clusters/puntos asignados para métricas internas (>=2 clusters y >=2 puntos).")

    # Probabilidades (fuerza de pertenencia)
    try:
        if hasattr(model, "probabilities_") and mask.any():
            metrics["mean_probability"] = float(np.mean(model.probabilities_[mask]))
        else:
            metrics["mean_probability"] = None
    except Exception as e:
        log_warning(f"No se pudo calcular mean_probability: {e}")
        metrics["mean_probability"] = None

    # Probabilidades (fuerza de pertenencia) y persistencia
    try:
        cp = getattr(model, "cluster_persistence_", None)

        # --- máscaras y validaciones ---
        mask_p = (labels >= 0)
        if cp is None or not mask_p.any():
            metrics["median_persistence_raw"] = 0.0
            metrics["median_persistence_main_weighted"] = 0.0
        else:
            labels_valid = labels[mask_p]
            cluster_ids, counts = np.unique(labels_valid, return_counts=True)

            total_points = counts.sum()
            if total_points == 0:
                metrics["median_persistence_raw"] = 0.0
                metrics["median_persistence_main_weighted"] = 0.0
            else:
                # --- preparar acceso a persistencias cp ---
                try:
                    cp_arr = np.asarray(cp, dtype=float)
                except Exception:
                    cp_arr = None

                def get_persistence(cid: int) -> float:
                    if isinstance(cp, dict):
                        return float(cp.get(int(cid), 0.0))
                    else:
                        if (cp_arr is not None) and (0 <= int(cid) < cp_arr.size):
                            return float(cp_arr[int(cid)])
                        return 0.0

                # -------- 1) RAW: mediana simple por cluster (sin ponderar) --------
                pers_all = np.asarray([get_persistence(cid) for cid in cluster_ids], dtype=float)
                metrics["median_persistence_raw"] = float(np.median(pers_all)) if pers_all.size > 0 else 0.0

                # -------- 2) MAIN_WEIGHTED: igual a _eval_hdbscan_once --------
                size_frac = counts / float(total_points)

                MAIN_FRAC = 0.005  # usa el mismo valor que en _eval_hdbscan_once
                main_mask = (size_frac >= MAIN_FRAC)

                main_cluster_ids = cluster_ids[main_mask]
                main_counts      = counts[main_mask]

                if main_cluster_ids.size == 0:
                    main_cluster_ids = cluster_ids
                    main_counts      = counts

                pers_main = []
                sizes_main = []
                for cid, csize in zip(main_cluster_ids, main_counts):
                    pers_main.append(get_persistence(cid))
                    sizes_main.append(float(csize))

                pers_main  = np.asarray(pers_main, dtype=float)
                sizes_main = np.asarray(sizes_main, dtype=float)

                if pers_main.size == 0:
                    metrics["median_persistence_main_weighted"] = 0.0
                else:
                    order = np.argsort(pers_main)
                    p_sorted = pers_main[order]
                    w_sorted = sizes_main[order]

                    w_cum = np.cumsum(w_sorted) / (w_sorted.sum() + 1e-12)
                    idx_m = int(np.searchsorted(w_cum, 0.5))
                    idx_m = min(max(idx_m, 0), p_sorted.size - 1)
                    metrics["median_persistence_main_weighted"] = float(p_sorted[idx_m])

    except Exception as e:
        log_warning(f"No se pudo calcular persistencias (raw/main_weighted): {e}")
        metrics["median_persistence_raw"] = None
        metrics["median_persistence_main_weighted"] = None


    # Logs finales
    log_info(
        "Métricas HDBSCAN -> "
        f"n_clusters={metrics['n_clusters']}, ruido={metrics['noise_ratio']:.2%}, "
        f"silhouette={metrics['silhouette']}, CH={metrics['calinski_harabasz']}, "
        f"DB={metrics['davies_bouldin']}, mean_prob={metrics['mean_probability']}, "
        f"median_persist_raw={metrics['median_persistence_raw']}, "
        f"median_persist_main_w={metrics['median_persistence_main_weighted']}"
    )

    return labels, model, metrics

### **Validar agrupaciones (ARI)**

In [54]:
def run_hdbscan_with_params(
    X,
    params: dict,
    consider_noise_in_mean_prob: bool = False,
    selection_method: str = "eom",
):
    """
    Ejecuta HDBSCAN con un dict de hiperparámetros y devuelve:
    (labels, model, mean_prob, median_persist, noise_pct, n_clusters)
    """
    mean_prob, median_persist, noise_pct, n_clusters, model = _eval_hdbscan_once(
        X,
        min_cluster_size=int(params["min_cluster_size"]),
        min_samples=int(params["min_samples"]),
        cluster_selection_epsilon=float(params["cluster_selection_epsilon"]),
        metric=params.get("metric", "euclidean"),
        consider_noise_in_mean_prob=consider_noise_in_mean_prob,
        selection_method=selection_method,
    )
    labels = model.labels_.copy()
    return labels, model, mean_prob, median_persist, noise_pct, n_clusters

Inyectamos a los datos pequeños ruidos gausianos, y volvemos a clusterizar, con el fin de determinar que tan cercanas son las asignaciones a la clusterización original. A este ruido también se le llama "jitter".

In [55]:
def evaluate_stability_random_jitter(
    X,
    best_params: dict,
    n_runs: int = 5,
    jitter_std: float = 0.01,
    random_state: int = 42,
) -> dict:
    rng = np.random.RandomState(random_state)
    sel = best_params.get("cluster_selection_method", "eom")
    log_info(f"[Stability-jitter] usando cluster_selection_method={sel}")

    # 0) Etiquetas base en los datos originales X
    base_labels, base_model, base_mean_prob, base_median_persist, base_noise_pct, base_n_clusters = (
        run_hdbscan_with_params(
            X,
            best_params,
            consider_noise_in_mean_prob=False,
            selection_method=sel,
        )
    )
    log_info(
        f"[Stability-jitter] Base -> n_clusters={base_n_clusters}, "
        f"noise={base_noise_pct:.2f}%, mean_prob={base_mean_prob:.3f}, "
        f"median_persist={base_median_persist:.3f}"
    )

    ari_scores = []
    n_clusters_list = []
    noise_list = []

    for i in range(n_runs):
        X_jit = X + rng.normal(scale=jitter_std, size=X.shape)

        labels, model, mean_prob, median_persist, noise_pct, n_clusters = run_hdbscan_with_params(
            X_jit,
            best_params,
            consider_noise_in_mean_prob=False,
            selection_method=sel,
        )

        n_clusters_list.append(n_clusters)
        noise_list.append(noise_pct)

        ari = adjusted_rand_score(base_labels, labels)
        ari_scores.append(ari)
        log_info(
            f"[Stability-jitter] Run {i}: n_clusters={n_clusters}, noise={noise_pct:.2f}%, "
            f"mean_prob={mean_prob:.3f}, median_persist={median_persist:.3f}, ARI vs base={ari:.3f}"
        )

    summary = {
        "mean_ari": float(np.mean(ari_scores)) if ari_scores else np.nan,
        "min_ari": float(np.min(ari_scores)) if ari_scores else np.nan,
        "max_ari": float(np.max(ari_scores)) if ari_scores else np.nan,
        "mean_n_clusters": float(np.mean(n_clusters_list)),
        "std_n_clusters": float(np.std(n_clusters_list)),
        "mean_noise_pct": float(np.mean(noise_list)),
        "std_noise_pct": float(np.std(noise_list)),
    }

    log_info(
        "[Stability-jitter] Resumen -> "
        f"mean_ari={summary['mean_ari']:.3f}, "
        f"n_clusters={summary['mean_n_clusters']:.1f} ± {summary['std_n_clusters']:.1f}, "
        f"noise={summary['mean_noise_pct']:.2f}% ± {summary['std_noise_pct']:.2f}%"
    )

    return summary

Ahora variamos levemente los hiperparámetros para visualizar los cambios en los cluster.

In [56]:
def evaluate_stability_multiresolution(
    X,
    best_params: dict,
    mcs_factors = (0.8, 1.0, 1.2),
    ms_factors  = (0.8, 1.0, 1.2),
    eps_factors = (0.5, 1.0, 1.5),
    consider_noise_in_mean_prob: bool = False,
) -> pd.DataFrame:
    """
    Evalúa estabilidad de la solución óptima frente a pequeñas variaciones
    en min_cluster_size, min_samples y cluster_selection_epsilon.

    Retorna un DataFrame con una fila por combinación y columnas:
      - min_cluster_size, min_samples, epsilon
      - mean_prob, median_persist, noise_pct, n_clusters
      - ari_vs_best
    """
    # 1) Etiquetas de referencia con los mejores parámetros
    sel = best_params.get("cluster_selection_method", "eom")

    base_labels, base_model, base_mean_prob, base_median_persist, base_noise_pct, base_n_clusters = (
        run_hdbscan_with_params(
            X,
            best_params,
            consider_noise_in_mean_prob=consider_noise_in_mean_prob,
            selection_method=sel,
        )
    )

    log_info(
        f"[Stability-multires] Referencia -> "
        f"mcs={best_params['min_cluster_size']}, ms={best_params['min_samples']}, "
        f"eps={best_params['cluster_selection_epsilon']:.3f}, metric={best_params.get('metric','euclidean')}, "
        f"mean_prob={base_mean_prob:.3f}, median_persist={base_median_persist:.3f}, "
        f"noise={base_noise_pct:.2f}%, n_clusters={base_n_clusters}"
    )

    rows = []
    N = X.shape[0]

    best_mcs = int(best_params["min_cluster_size"])
    best_ms  = int(best_params["min_samples"])
    best_eps = float(best_params["cluster_selection_epsilon"])
    metric   = best_params.get("metric", "euclidean")

    # Construimos pequeñas variaciones alrededor del óptimo
    for fmcs in mcs_factors:
        for fms in ms_factors:
            for feps in eps_factors:
                mcs = max(5, int(best_mcs * fmcs))
                ms  = max(5, int(best_ms * fms))
                eps = max(0.0, best_eps * feps)

                params = {
                    "min_cluster_size": mcs,
                    "min_samples": ms,
                    "cluster_selection_epsilon": eps,
                    "metric": metric,
                }

                labels, model, mean_prob, median_persist, noise_pct, n_clusters = run_hdbscan_with_params(
                    X,
                    params,
                    consider_noise_in_mean_prob=consider_noise_in_mean_prob,
                    selection_method=sel,
                )

                ari = adjusted_rand_score(base_labels, labels)

                log_info(
                    f"[Stability-multires] mcs={mcs}, ms={ms}, eps={eps:.3f} "
                    f"-> mean_prob={mean_prob:.3f}, median_persist={median_persist:.3f}, "
                    f"noise={noise_pct:.2f}%, n_clusters={n_clusters}, ARI vs best={ari:.3f}"
                )

                rows.append({
                    "min_cluster_size": mcs,
                    "min_samples": ms,
                    "cluster_selection_epsilon": eps,
                    "mean_prob": mean_prob,
                    "median_persist": median_persist,
                    "noise_pct": noise_pct,
                    "n_clusters": n_clusters,
                    "ari_vs_best": ari,
                })

    df_res = pd.DataFrame(rows)
    return df_res

Creamos una función helper que nos permite implementar las dos funciones ARI para la validación de la clusterización

In [57]:
def run_hdbscan_stability_checks(
    X,
    best_params: dict,
    output_dir: Path,
    consider_noise_in_mean_prob: bool = False,
    random_state: int = RANDOM_STATE,
    n_jitter_runs: int = 5,
    jitter_std: float = 0.01,
) -> Tuple[dict, pd.DataFrame]:
    """
    Ejecuta las pruebas de estabilidad:
      - jitter aleatorio sobre X
      - variaciones pequeñas en min_cluster_size, min_samples y epsilon

    Guarda los resultados en disco y retorna:
      (summary_jitter, df_multiresolution)
    """
    output_dir.mkdir(parents=True, exist_ok=True)

    # Asegurar formato numpy
    X_arr = X.values if isinstance(X, pd.DataFrame) else X

    # 1) Estabilidad frente a jitter
    stability_jitter = evaluate_stability_random_jitter(
        X=X_arr,
        best_params=best_params,
        n_runs=n_jitter_runs,
        jitter_std=jitter_std,
        random_state=random_state,
    )

    # 2) Estabilidad frente a perturbaciones de hiperparámetros
    stability_multires_df = evaluate_stability_multiresolution(
        X=X_arr,
        best_params=best_params,
        consider_noise_in_mean_prob=consider_noise_in_mean_prob,
    )

    # Guardar a disco
    jitter_path = output_dir / "stability_jitter.json"
    with jitter_path.open("w", encoding="utf-8") as f:
        json.dump(stability_jitter, f, ensure_ascii=False, indent=2)
    log_info(f"[Stability] Resumen jitter guardado en: {jitter_path}")

    multires_path = output_dir / "stability_multiresolution.csv"
    save_dataframe_to_csv(stability_multires_df, multires_path, index=False)
    log_info(f"[Stability] Resultados multiresolution guardados en: {multires_path}")

    return stability_jitter, stability_multires_df

### **Reasignación de Ruido a Clusters**

In [58]:
def reassign_noise_to_nearest_centroid(
    X,
    labels,
    min_cluster_size_for_centroid: int = 5,
    metric: str = "euclidean"
):
    """
    Reasigna los puntos de ruido (-1) al cluster cuyo centroide esté más cerca.

    Devuelve:
      - new_labels: array de etiquetas sin -1
      - reassignment_confidence: array de floats en [0,1] (1 = alta confianza).
      - was_noise: array booleano (True si el punto era ruido original).
    """
    X_arr = X.values if isinstance(X, pd.DataFrame) else np.asarray(X)
    labels = np.asarray(labels)

    n_samples, n_features = X_arr.shape

    was_noise = (labels == -1)
    non_noise = ~was_noise

    if was_noise.sum() == 0:
        # No hay ruido, nada que hacer
        return labels.copy(), np.ones_like(labels, dtype=float), was_noise

    cluster_ids = np.unique(labels[non_noise])

    # centroides por cluster
    centroids = []
    valid_cluster_ids = []

    for k in cluster_ids:
        idx = (labels == k)
        if idx.sum() < min_cluster_size_for_centroid:
            # clusters demasiado pequeños se pueden saltar
            continue
        centroids.append(X_arr[idx].mean(axis=0))
        valid_cluster_ids.append(k)

    centroids = np.vstack(centroids)
    valid_cluster_ids = np.array(valid_cluster_ids)

    # distancias de puntos ruido a cada centroide
    X_noise = X_arr[was_noise]
    dist = pairwise_distances(X_noise, centroids, metric=metric)

    # cluster más cercano
    nearest_idx = dist.argmin(axis=1)
    assigned_clusters = valid_cluster_ids[nearest_idx]

    # confianza (basada en distancia al 1er y 2do vecino)
    sorted_dist = np.sort(dist, axis=1)
    d1 = sorted_dist[:, 0]
    d2 = sorted_dist[:, 1] if dist.shape[1] > 1 else (sorted_dist[:, 0] + 1e-8)

    # 1 - d1/(d1 + d2) -> si d1 << d2, confianza alta
    confidence_noise = 1.0 - d1 / (d1 + d2 + 1e-8)

    new_labels = labels.copy()
    new_labels[was_noise] = assigned_clusters

    reassignment_confidence = np.ones_like(labels, dtype=float)
    reassignment_confidence[was_noise] = confidence_noise

    return new_labels, reassignment_confidence, was_noise

### **Crear Macro Clusters**

In [59]:
def create_macro_clusters(
    X,
    labels_raw: np.ndarray,
    max_macro_clusters: int = 10,
    linkage_method: str = "ward",
    min_cluster_size_for_centroid: int = 5
) -> Dict[int, int]:
    """
    Agrupa clusters HDBSCAN en macro-clusters jerárquicos.
    
    Solo considera clusters válidos (>=0), el ruido (-1) no participa
    en el cálculo de centroides.
    
    Args:
        X: Datos escalados (DataFrame o array).
        labels_raw: Etiquetas HDBSCAN originales (-1 = ruido).
        max_macro_clusters: Número máximo de macro-clusters a generar.
        linkage_method: Método de linkage para AgglomerativeClustering 
                       ('ward', 'complete', 'average', 'single').
        min_cluster_size_for_centroid: Tamaño mínimo de cluster para calcular centroide.
    
    Returns:
        Diccionario {cluster_id: macro_cluster_id}
    """
    log_info(f"Creando macro-clusters con max={max_macro_clusters}, linkage={linkage_method}")
    
    X_arr = X.values if isinstance(X, pd.DataFrame) else np.asarray(X)
    labels_raw = np.asarray(labels_raw)
    
    # Filtrar solo puntos válidos (no ruido)
    valid_mask = labels_raw >= 0
    
    if valid_mask.sum() == 0:
        log_warning("No hay clusters válidos para crear macro-clusters.")
        return {}
    
    # Obtener IDs de clusters únicos
    cluster_ids = np.unique(labels_raw[valid_mask])
    
    # Calcular centroides por cluster
    centroids = []
    valid_cluster_ids = []
    
    for k in cluster_ids:
        cluster_mask = (labels_raw == k)
        if cluster_mask.sum() < min_cluster_size_for_centroid:
            continue
        centroids.append(X_arr[cluster_mask].mean(axis=0))
        valid_cluster_ids.append(k)
    
    if len(centroids) == 0:
        log_warning("No hay suficientes clusters válidos para crear macro-clusters.")
        return {}
    
    centroids = np.vstack(centroids)
    valid_cluster_ids = np.array(valid_cluster_ids)
    
    n_clusters = len(valid_cluster_ids)
    n_macro = min(max_macro_clusters, n_clusters)
    
    log_info(f"Clusters válidos: {n_clusters}, Macro-clusters objetivo: {n_macro}")
    
    if n_clusters <= n_macro:
        # Cada cluster es su propio macro-cluster
        log_info("Número de clusters <= max_macro_clusters. Cada cluster es un macro-cluster.")
        return {int(k): int(k) for k in valid_cluster_ids}
    
    # Aplicar Agglomerative Clustering sobre centroides
    agg = AgglomerativeClustering(
        n_clusters=n_macro,
        linkage=linkage_method
    )
    macro_labels = agg.fit_predict(centroids)
    
    # Crear mapeo cluster_id -> macro_cluster_id
    cluster_to_macro = {int(cid): int(mid) for cid, mid in zip(valid_cluster_ids, macro_labels)}
    
    # Log de distribución de macro-clusters (cuántos clusters hay en cada macro)
    macro_counts = {}
    for mid in macro_labels:
        mid_int = int(mid)  # Convertir a int nativo de Python
        macro_counts[mid_int] = macro_counts.get(mid_int, 0) + 1
    
    log_info(f"Macro-clusters creados: {len(set(macro_labels))}")
    
    # Mensaje más claro: "Macro-cluster X contiene Y clusters"
    distribution_msg = ", ".join([f"Macro {k}: {v} clusters" for k, v in sorted(macro_counts.items())])
    log_info(f"Composición: {distribution_msg}")
    
    return cluster_to_macro

### **Gráfico de clusters**

In [60]:
import plotly.io as pio

def _sorted_label_strings_numeric_first(labels: np.ndarray) -> list:
    """
    Devuelve las etiquetas como strings, ordenadas numéricamente si es posible.
    Primero los que se pueden parsear a número (orden numérico), luego el resto (orden alfabético).
    """
    labels_str = labels.astype(str)
    uniq = sorted(set(labels_str), key=lambda s: (
        # clave primaria: ¿es numérico?
        not s.replace('.', '', 1).lstrip('-').isdigit(),
        # clave secundaria: valor numérico si aplica (int primero, luego float), si no, el string
        float(s) if s.replace('.', '', 1).lstrip('-').isdigit() else s
    ))
    return [str(u) for u in uniq]

def _heatmap_colors(n: int, cmap_name: str = "hot") -> list:
    """
    Genera n colores (hex) desde un colormap tipo 'heatmap' (por defecto: 'hot').
    """
    # Use the recommended way to get colormaps in newer Matplotlib versions
    cmap = plt.get_cmap(cmap_name)
    # espaciado uniforme en [0,1]; evitar extremos idénticos si n=1
    vals = np.linspace(0.1, 0.9, num=max(n, 1))
    return [mcolors.to_hex(cmap(v)) for v in vals]

def _choose_plotly_renderer() -> str:
    """Elige un renderer adecuado según el entorno de ejecución."""
    # Databricks
    if os.environ.get("DATABRICKS_RUNTIME_VERSION"):
        return "databricks"
    # Google Colab
    if os.environ.get("COLAB_RELEASE_TAG"):
        return "colab"
    # Jupyter (ZMQ)
    try:
        from IPython import get_ipython  # noqa: WPS433 (import inside)
        ip = get_ipython()
        if ip and "ZMQInteractiveShell" in str(type(ip)):
            return "notebook_connected"
    except Exception:
        pass
    # Fallback: abre en navegador
    return "browser"


def plot_clusters_tsne(reduced_df: pd.DataFrame, cluster_labels: np.ndarray) -> None:
    """
    Grafica clusters en 2D o 3D (según columnas de `reduced_df`) para visualización con t-SNE.
    - Colores: paleta 'hot' (heatmap).
    - Leyenda: orden numérico de etiquetas.
    - Ejes: x, y (y z si es 3D).
    Además guarda un HTML en DIR_MODELS/clusters_tsne.html
    """
    # 0) Comprobaciones básicas
    if not PLOTLY_AVAILABLE:
        log_warning(PLOTLY_IMPORT_WARNING or "Plotly no disponible.")
        return
    if reduced_df is None or reduced_df.empty:
        log_warning("reduced_df está vacío. No se puede graficar.")
        return
    if len(reduced_df) != len(cluster_labels):
        log_warning(f"Tamaño inconsistente: reduced_df={len(reduced_df)}, labels={len(cluster_labels)}")
        return

    # 1) Dimensiones válidas
    n_dims = reduced_df.shape[1]
    if n_dims not in (2, 3):
        log_warning(f"Dimensionalidad no soportada para graficar: {n_dims}. Se requiere 2 o 3 dimensiones.")
        return

    # 2) Columnas x,y,(z)
    axis_names = ["x", "y"] if n_dims == 2 else ["x", "y", "z"]
    plot_df = reduced_df.copy()
    plot_df.columns = axis_names

    # 3) Etiquetas ordenadas y colores
    labels_str = np.asarray(cluster_labels).astype(str)
    ordered_labels = _sorted_label_strings_numeric_first(labels_str)
    n_labels = len(ordered_labels)
    colors = _heatmap_colors(n_labels, cmap_name="hot")
    color_map = {lab: colors[i] for i, lab in enumerate(ordered_labels)}
    plot_df["cluster"] = labels_str

    # 4) Construcción de la figura
    if n_dims == 2:
        log_info("Generando gráfico 2D de clusters (t-SNE).")
        fig = px.scatter(
            plot_df,
            x="x", y="y",
            color="cluster",
            color_discrete_map=color_map,
            category_orders={"cluster": ordered_labels},
            title="Clusters (t-SNE 2D)",
            width=900, height=700
        )
        fig.update_layout(xaxis_title="x", yaxis_title="y")
    else:
        log_info("Generando gráfico 3D de clusters (t-SNE).")
        fig = px.scatter_3d(
            plot_df,
            x="x", y="y", z="z",
            color="cluster",
            color_discrete_map=color_map,
            category_orders={"cluster": ordered_labels},
            title="Clusters (t-SNE 3D)",
            width=1000, height=750
        )
        fig.update_layout(scene=dict(
            xaxis_title="x",
            yaxis_title="y",
            zaxis_title="z"
        ))

    # 5) Elegir renderer según entorno
    try:
        renderer = _choose_plotly_renderer()
        pio.renderers.default = renderer
        log_info(f"Plotly renderer seleccionado: {renderer}")
    except Exception as e:
        log_warning(f"No se pudo fijar renderer automáticamente: {e}")

    # 6) Mostrar en notebook / entorno interactivo si es posible
    try:
        # Databricks: displayHTML con HTML de Plotly
        if os.environ.get("DATABRICKS_RUNTIME_VERSION") and "displayHTML" in globals():
            html = fig.to_html(include_plotlyjs="cdn", full_html=False)
            displayHTML(html)  # type: ignore  # Databricks utility
        else:
            # Jupyter / Colab / otros
            fig.show()
    except Exception as e:
        log_warning(f"No se pudo mostrar inline el gráfico: {e}")

    # 7) Guardar HTML en disco para consulta posterior
    try:
        out_html = DIR_MODELS / "clusters_tsne.html"
        out_html.parent.mkdir(parents=True, exist_ok=True)
        html = fig.to_html(include_plotlyjs="cdn", full_html=True)
        with open(out_html, "w", encoding="utf-8") as f:
            f.write(html)
        log_info(f"Gráfico guardado en: {out_html.resolve()}")
    except Exception as e:
        log_warning(f"No se pudo guardar el HTML del gráfico: {e}")

    fig.show()
    log_info(f"Gráfico {n_dims}D generado con paleta 'hot' y leyenda ordenada numéricamente.")

### **Retornar csv limpio con etiquetas de clusters**

In [61]:
def write_clustered_full_csv(
    original_df: pd.DataFrame,
    labels_raw: np.ndarray,
    labels_filled: np.ndarray,
    was_noise: np.ndarray,
    noise_confidence: np.ndarray,
    sampled_index: pd.Index,
    out_path: Path,
    id_col: str = "RowID",
    original_columns: Optional[List[str]] = None
) -> Path:
    """
    Devuelve a disco un CSV con TODO el dataset original + columnas de clusterización:
      - cluster_label: etiqueta final (ruido reasignado, no muestreado = -2)
      - cluster_label_raw: etiqueta HDBSCAN original (ruido = -1, no muestreado = -2)
      - is_noise: True si la fila era ruido en HDBSCAN
      - is_in_sample: True si la fila estuvo en la muestra usada para clusterizar
      - noise_reassign_confidence: confianza en la reasignación (NaN si no era ruido)

    Solo conserva las columnas "originales" para evitar que se cuelen ILR/Log1p/etc.
    """
    log_info(f"Escribiendo CSV completo con etiquetas en: {out_path}")

    # 1) Partimos de las columnas originales
    if original_columns is None:
        drop_patterns = ("_ILR_", "Log1p_", "FAMD")
        keep_cols = [
            c for c in original_df.columns
            if not any(p in c for p in drop_patterns)
        ]
        if id_col not in keep_cols and id_col in original_df.columns:
            keep_cols = [id_col] + keep_cols
        base_df = original_df.loc[:, keep_cols].copy()
    else:
        base_df = original_df.loc[:, [c for c in original_columns if c in original_df.columns]].copy()

    # 2) Series de etiquetas para TODAS las filas
    #    -2 si no está en la muestra
    cluster_label_raw = pd.Series(-2, index=base_df.index, dtype=int)
    cluster_label     = pd.Series(-2, index=base_df.index, dtype=int)
    was_noise_full    = pd.Series(False, index=base_df.index, dtype=bool)
    noise_conf_full   = pd.Series(np.nan, index=base_df.index, dtype=float)

    # Aseguramos que índice de sampled_index está en el mismo espacio que base_df.index
    # (en tu pipeline esto se cumple: ambos usan RowID como índice)
    cluster_label_raw.loc[sampled_index] = labels_raw.astype(int)
    cluster_label.loc[sampled_index]     = labels_filled.astype(int)
    was_noise_full.loc[sampled_index]    = was_noise.astype(bool)
    noise_conf_full.loc[sampled_index]   = noise_confidence.astype(float)

    is_in_sample = base_df.index.isin(sampled_index)
    is_noise = cluster_label_raw.eq(-1)

    # 3) Añadir columnas de salida
    base_df["cluster_label"] = cluster_label
    base_df["cluster_label_raw"] = cluster_label_raw
    base_df["is_noise"] = is_noise
    base_df["is_in_sample"] = is_in_sample
    base_df["noise_reassign_confidence"] = noise_conf_full

    # 4) Orden opcional: RowID primero si existe
    if id_col in base_df.columns:
        ordered_cols = [id_col] + [c for c in base_df.columns if c != id_col]
        base_df = base_df[ordered_cols]

    # 5) Guardar
    out_path.parent.mkdir(parents=True, exist_ok=True)
    base_df.to_csv(out_path, index=False, encoding="utf-8-sig")
    log_info(f"Archivo guardado: {out_path} (filas={base_df.shape[0]}, cols={base_df.shape[1]})")
    return out_path


def write_clustered_full_csv_with_macro(
    original_df: pd.DataFrame,
    labels_raw: np.ndarray,
    labels_filled: np.ndarray,
    was_noise: np.ndarray,
    noise_confidence: np.ndarray,
    sampled_index: pd.Index,
    macro_cluster_map: Dict[int, int],
    out_path: Path,
    id_col: str = "RowID",
    original_columns: Optional[List[str]] = None
) -> Path:
    """
    Devuelve a disco un CSV con TODO el dataset original + columnas de clusterización
    incluyendo la columna de macro-clusters:
    
      - cluster_label: etiqueta final (ruido reasignado, no muestreado = -2)
      - cluster_label_raw: etiqueta HDBSCAN original (ruido = -1, no muestreado = -2)
      - is_noise: True si la fila era ruido en HDBSCAN
      - is_in_sample: True si la fila estuvo en la muestra usada para clusterizar
      - noise_reassign_confidence: confianza en la reasignación (NaN si no era ruido)
      - macro_cluster: etiqueta del macro-cluster (agrupación jerárquica de clusters)

    Solo conserva las columnas "originales" para evitar que se cuelen ILR/Log1p/etc.
    """
    log_info(f"Escribiendo CSV completo con etiquetas y macro-clusters en: {out_path}")

    # 1) Partimos de las columnas originales
    if original_columns is None:
        drop_patterns = ("_ILR_", "Log1p_", "FAMD")
        keep_cols = [
            c for c in original_df.columns
            if not any(p in c for p in drop_patterns)
        ]
        if id_col not in keep_cols and id_col in original_df.columns:
            keep_cols = [id_col] + keep_cols
        base_df = original_df.loc[:, keep_cols].copy()
    else:
        base_df = original_df.loc[:, [c for c in original_columns if c in original_df.columns]].copy()

    # 2) Series de etiquetas para TODAS las filas
    #    -2 si no está en la muestra
    cluster_label_raw = pd.Series(-2, index=base_df.index, dtype=int)
    cluster_label     = pd.Series(-2, index=base_df.index, dtype=int)
    was_noise_full    = pd.Series(False, index=base_df.index, dtype=bool)
    noise_conf_full   = pd.Series(np.nan, index=base_df.index, dtype=float)
    macro_cluster     = pd.Series(-2, index=base_df.index, dtype=int)

    # Aseguramos que índice de sampled_index está en el mismo espacio que base_df.index
    cluster_label_raw.loc[sampled_index] = labels_raw.astype(int)
    cluster_label.loc[sampled_index]     = labels_filled.astype(int)
    was_noise_full.loc[sampled_index]    = was_noise.astype(bool)
    noise_conf_full.loc[sampled_index]   = noise_confidence.astype(float)
    
    # Asignar macro-clusters basándose en cluster_label (etiquetas finales con ruido reasignado)
    for idx in sampled_index:
        cl = int(cluster_label.loc[idx])
        if cl in macro_cluster_map:
            macro_cluster.loc[idx] = macro_cluster_map[cl]
        else:
            # Para clusters no mapeados (ej: ruido que no estaba en el mapeo original)
            macro_cluster.loc[idx] = -1

    is_in_sample = base_df.index.isin(sampled_index)
    is_noise = cluster_label_raw.eq(-1)

    # 3) Añadir columnas de salida
    base_df["cluster_label"] = cluster_label
    base_df["cluster_label_raw"] = cluster_label_raw
    base_df["is_noise"] = is_noise
    base_df["is_in_sample"] = is_in_sample
    base_df["noise_reassign_confidence"] = noise_conf_full
    base_df["macro_cluster"] = macro_cluster

    # 4) Orden opcional: RowID primero si existe
    if id_col in base_df.columns:
        ordered_cols = [id_col] + [c for c in base_df.columns if c != id_col]
        base_df = base_df[ordered_cols]

    # 5) Guardar
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    base_df.to_csv(out_path, index=False, encoding="utf-8-sig")
    
    # 6) Log de estadísticas de macro-clusters
    macro_stats = base_df[base_df["is_in_sample"]]["macro_cluster"].value_counts().sort_index()
    log_info(f"Archivo guardado: {out_path} (filas={base_df.shape[0]}, cols={base_df.shape[1]})")
    log_info(f"Distribución de macro-clusters en datos:\n{macro_stats.to_string()}")
    
    return out_path

### **Perfiles de Clustering**

In [62]:
!pip install xlsxwriter

In [ ]:
def generate_cluster_profiles_excel(
    labeled_df: pd.DataFrame,
    output_xlsx: Path,
    id_col: str = "RowID",
    cluster_col: str = "cluster_label",
    exclude_noise: bool = True,       # excluye -1
    exclude_unseen: bool = True,      # excluye -2
    top_k_modes: int = 5,             # top-k categorías por variable categórica
    categorical_cols: Optional[List[str]] = None # [MODIFICADO] Lista explícita de categóricas
) -> Path:
    """
    Crea un Excel con perfilamiento de clusters usando SOLO variables originales.
    [MODIFICADO]: Acepta lista explicita de columnas categoricas para evitar inferencia numerica de IDs.
    """
    # -------- 1) Seleccionar SOLO columnas originales ----------
    # Columnas meta que NO deben perfilarse
    meta_cols = {
        id_col, cluster_col,
        "is_noise", "is_in_sample", "is_clustered",
        "cluster_label_raw", "noise_reassign_confidence"
    }
    # Prefijos de columnas derivadas que NO son originales:
    derived_prefixes = (
        "GENERO_ILR_", "TARIFA_ILR_", "EDAD_ILR_",
        "FAMD", "Log1p_"
    )

    orig_cols = []
    for c in labeled_df.columns:
        if c in meta_cols:
            continue
        if any(c.startswith(p) for p in derived_prefixes):
            continue
        orig_cols.append(c)

    if cluster_col not in labeled_df.columns:
        raise ValueError(f"No encuentro la columna de cluster '{cluster_col}' en el DataFrame.")

    # Trabajamos sobre una copia “solo originales + meta”
    cols_for_work = [id_col, cluster_col] + [c for c in orig_cols if c in labeled_df.columns]
    work = labeled_df[cols_for_work].copy()

    # -------- 2) Filtrado de filas por reglas de negocio ----------
    mask = pd.Series(True, index=work.index)
    if exclude_noise:
        mask &= (work[cluster_col] != -1)
    if exclude_unseen:
        mask &= (work[cluster_col] != -2)

    df_in = work.loc[mask].copy()
    if df_in.empty:
        raise ValueError("No quedan filas para perfilar (¿todos ruido y/o no muestreados?).")

    # -------- 3) Tipos: Definicion explicita vs inferida ----------
    #Logica para forzar categoricas
    if categorical_cols:
        # Aseguramos que existan en el df
        valid_cats = [c for c in categorical_cols if c in df_in.columns]
        # Forzamos tipo object/string para evitar que se interpreten como numeros
        for c in valid_cats:
            df_in[c] = df_in[c].astype(str)
        
        cat_cols_final = valid_cats
        # Las numericas son las que estan en orig_cols pero NO en cat_cols_final (y no son id/cluster)
        num_cols_final = [
            c for c in orig_cols 
            if c not in cat_cols_final 
            and c not in (id_col, cluster_col)
            and c in df_in.columns
        ]
    else:
        # Fallback a inferencia (comportamiento anterior)
        num_cols_final = df_in.select_dtypes(include=np.number).columns.tolist()
        num_cols_final = [c for c in num_cols_final if c not in (id_col, cluster_col)]
        cat_cols_final = [c for c in orig_cols if c not in num_cols_final and c in df_in.columns]

    # -------- 4) Estadísticos GLOBAL (para hoja Global Summary & robust-z) ----------
    def iqr(s: pd.Series) -> float:
        return np.nanpercentile(s, 75) - np.nanpercentile(s, 25)

    def mad_raw(s: pd.Series) -> float:
        med = np.nanmedian(s)
        return np.nanmedian(np.abs(s - med))

    global_num = []
    for col in num_cols_final:
        s = pd.to_numeric(df_in[col], errors="coerce")
        global_num.append({
            "variable": col,
            "n": int(s.notna().sum()),
            "median": float(np.nanmedian(s)),
            "IQR": float(iqr(s)),
            "mean": float(np.nanmean(s)),
            "std": float(np.nanstd(s, ddof=1)) if s.notna().sum() > 1 else np.nan,
            "min": float(np.nanmin(s)),
            "max": float(np.nanmax(s)),
            "MAD_raw": float(mad_raw(s)),
        })
    global_num_df = pd.DataFrame(global_num).sort_values("variable")

    # Categóricas globales (top-k)
    global_cat_rows = []
    for col in cat_cols_final:
        vc = df_in[col].astype("category").value_counts(dropna=False)
        total = int(vc.sum())
        # top-k
        top = vc.head(top_k_modes)
        for cat_val, cnt in top.items():
            global_cat_rows.append({
                "variable": col,
                "category": str(cat_val),
                "count": int(cnt),
                "pct": float(cnt / total * 100.0) if total > 0 else np.nan,
            })
    global_cat_df = pd.DataFrame(global_cat_rows).sort_values(["variable", "count"], ascending=[True, False])

    # -------- 5) Estadísticos por cluster (numéricos) + robust-z ----------
    # robust-z = (median_cluster - median_global) / (MAD_raw_global + 1e-9)
    global_median_map = dict(zip(global_num_df["variable"], global_num_df["median"]))
    global_mad_map = dict(zip(global_num_df["variable"], global_num_df["MAD_raw"]))

    clust_num_rows = []
    for clu, g in df_in.groupby(cluster_col, sort=True):
        for col in num_cols_final:
            s = pd.to_numeric(g[col], errors="coerce")
            n = int(s.notna().sum())
            med = float(np.nanmedian(s)) if n > 0 else np.nan
            row = {
                "cluster": int(clu),
                "variable": col,
                "n": n,
                "median": med,
                "IQR": float(iqr(s)) if n > 0 else np.nan,
                "mean": float(np.nanmean(s)) if n > 0 else np.nan,
                "std": float(np.nanstd(s, ddof=1)) if n > 1 else np.nan,
                "min": float(np.nanmin(s)) if n > 0 else np.nan,
                "max": float(np.nanmax(s)) if n > 0 else np.nan,
            }
            g_median = global_median_map.get(col, np.nan)
            g_mad = global_mad_map.get(col, np.nan)
            denom = (g_mad if pd.notna(g_mad) and g_mad > 0 else 1e-9)
            row["robust_z_median"] = float((med - g_median) / denom) if pd.notna(med) and pd.notna(g_median) else np.nan
            clust_num_rows.append(row)

    clusters_overview_df = pd.DataFrame(clust_num_rows).sort_values(["cluster", "variable"])

    # -------- 6) Categóricas por cluster (top-k por variable) ----------
    clust_cat_rows = []
    for clu, g in df_in.groupby(cluster_col, sort=True):
        for col in cat_cols_final:
            s = g[col].astype("category")
            vc = s.value_counts(dropna=False)
            total = int(vc.sum())
            top = vc.head(top_k_modes)
            for cat_val, cnt in top.items():
                clust_cat_rows.append({
                    "cluster": int(clu),
                    "variable": col,
                    "category": str(cat_val),
                    "count": int(cnt),
                    "pct": float(cnt / total * 100.0) if total > 0 else np.nan,
                })

    categorical_overview_df = pd.DataFrame(clust_cat_rows).sort_values(["cluster", "variable", "count"], ascending=[True, True, False])

    # -------- 7) Exportar a Excel ----------
    output_xlsx = Path(output_xlsx)
    output_xlsx.parent.mkdir(parents=True, exist_ok=True)

    with pd.ExcelWriter(output_xlsx, engine="xlsxwriter") as writer:
        # Global Summary
        startrow = 0
        global_num_df.to_excel(writer, sheet_name="Global Summary", index=False, startrow=startrow)
        ws = writer.sheets["Global Summary"]
        ws.write(startrow, 0, "Global numeric summary (medians, IQR, mean, std, min, max, MAD_raw)")

        # Espacio y categóricas
        startrow = len(global_num_df) + 3
        ws.write(startrow, 0, f"Global categorical top-{top_k_modes} per variable")
        global_cat_df.to_excel(writer, sheet_name="Global Summary", index=False, startrow=startrow+1)

        # Clusters Overview (numéricas)
        clusters_overview_df.to_excel(writer, sheet_name="Clusters_Overview", index=False)
        ws2 = writer.sheets["Clusters_Overview"]
        ws2.freeze_panes(1, 0)

        # Categóricas por cluster
        categorical_overview_df.to_excel(writer, sheet_name="Categorical_Overview", index=False)
        ws3 = writer.sheets["Categorical_Overview"]
        ws3.freeze_panes(1, 0)

    log_info(f"Perfiles de clusters exportados a: {output_xlsx}")
    return output_xlsx


### **Perfiles de Macro Clusters**

In [64]:
def generate_macro_cluster_profiles_excel(
    labeled_df: pd.DataFrame,
    output_xlsx: Path,
    id_col: str = "RowID",
    macro_col: str = "macro_cluster",
    cluster_col: str = "cluster_label",
    exclude_noise: bool = True,       # excluye -1
    exclude_unseen: bool = True,      # excluye -2
    top_k_modes: int = 5,             # top-k categorías por variable categórica
    categorical_cols: Optional[List[str]] = None
) -> Path:
    """
    Crea un Excel con perfilamiento de MACRO-CLUSTERS usando SOLO variables originales.
    
    Similar a generate_cluster_profiles_excel pero agrupa por macro_cluster.
    Incluye una hoja adicional mostrando qué clusters pertenecen a cada macro-cluster.
    
    Args:
        labeled_df: DataFrame con datos y columnas de clustering
        output_xlsx: Ruta del archivo Excel de salida
        id_col: Nombre de la columna ID
        macro_col: Nombre de la columna de macro-clusters
        cluster_col: Nombre de la columna de clusters (para mostrar composición)
        exclude_noise: Excluir filas con macro_cluster = -1
        exclude_unseen: Excluir filas con macro_cluster = -2
        top_k_modes: Top-k categorías a mostrar por variable categórica
        categorical_cols: Lista explícita de columnas categóricas
    
    Returns:
        Path al archivo Excel generado
    """
    log_info(f"Generando perfiles de MACRO-CLUSTERS en: {output_xlsx}")
    
    # -------- 1) Seleccionar SOLO columnas originales ----------
    meta_cols = {
        id_col, macro_col, cluster_col,
        "is_noise", "is_in_sample", "is_clustered",
        "cluster_label_raw", "noise_reassign_confidence"
    }
    
    derived_prefixes = (
        "GENERO_ILR_", "TARIFA_ILR_", "EDAD_ILR_",
        "FAMD", "Log1p_"
    )
    
    orig_cols = []
    for c in labeled_df.columns:
        if c in meta_cols:
            continue
        if any(c.startswith(p) for p in derived_prefixes):
            continue
        orig_cols.append(c)
    
    if macro_col not in labeled_df.columns:
        raise ValueError(f"No encuentro la columna de macro-cluster '{macro_col}' en el DataFrame.")
    
    # Trabajamos sobre una copia
    cols_for_work = [id_col, macro_col, cluster_col] + [c for c in orig_cols if c in labeled_df.columns]
    cols_for_work = [c for c in cols_for_work if c in labeled_df.columns]
    work = labeled_df[cols_for_work].copy()
    
    # -------- 2) Filtrado de filas ----------
    mask = pd.Series(True, index=work.index)
    if exclude_noise:
        mask &= (work[macro_col] != -1)
    if exclude_unseen:
        mask &= (work[macro_col] != -2)
    
    df_in = work.loc[mask].copy()
    if df_in.empty:
        raise ValueError("No quedan filas para perfilar macro-clusters.")
    
    # -------- 3) Tipos: Definición explícita vs inferida ----------
    if categorical_cols:
        valid_cats = [c for c in categorical_cols if c in df_in.columns]
        for c in valid_cats:
            df_in[c] = df_in[c].astype(str)
        cat_cols_final = valid_cats
        num_cols_final = [
            c for c in orig_cols 
            if c not in cat_cols_final 
            and c not in (id_col, macro_col, cluster_col)
            and c in df_in.columns
        ]
    else:
        num_cols_final = df_in.select_dtypes(include=np.number).columns.tolist()
        num_cols_final = [c for c in num_cols_final if c not in (id_col, macro_col, cluster_col)]
        cat_cols_final = [c for c in orig_cols if c not in num_cols_final and c in df_in.columns]
    
    # -------- 4) Funciones auxiliares ----------
    def iqr(s: pd.Series) -> float:
        return np.nanpercentile(s, 75) - np.nanpercentile(s, 25)
    
    def mad_raw(s: pd.Series) -> float:
        med = np.nanmedian(s)
        return np.nanmedian(np.abs(s - med))
    
    # -------- 5) Estadísticos GLOBAL ----------
    global_num = []
    for col in num_cols_final:
        s = pd.to_numeric(df_in[col], errors="coerce")
        global_num.append({
            "variable": col,
            "n": int(s.notna().sum()),
            "median": float(np.nanmedian(s)),
            "IQR": float(iqr(s)),
            "mean": float(np.nanmean(s)),
            "std": float(np.nanstd(s, ddof=1)) if s.notna().sum() > 1 else np.nan,
            "min": float(np.nanmin(s)),
            "max": float(np.nanmax(s)),
            "MAD_raw": float(mad_raw(s)),
        })
    global_num_df = pd.DataFrame(global_num).sort_values("variable") if global_num else pd.DataFrame()
    
    # Categóricas globales
    global_cat_rows = []
    for col in cat_cols_final:
        vc = df_in[col].astype("category").value_counts(dropna=False)
        total = int(vc.sum())
        top = vc.head(top_k_modes)
        for cat_val, cnt in top.items():
            global_cat_rows.append({
                "variable": col,
                "category": str(cat_val),
                "count": int(cnt),
                "pct": float(cnt / total * 100.0) if total > 0 else np.nan,
            })
    global_cat_df = pd.DataFrame(global_cat_rows).sort_values(["variable", "count"], ascending=[True, False]) if global_cat_rows else pd.DataFrame()
    
    # -------- 6) Composición de macro-clusters ----------
    composition_rows = []
    for macro_id in sorted(df_in[macro_col].unique()):
        macro_data = df_in[df_in[macro_col] == macro_id]
        clusters_in_macro = sorted(macro_data[cluster_col].unique())
        n_empresas = len(macro_data)
        
        composition_rows.append({
            "macro_cluster": int(macro_id),
            "clusters_incluidos": str(clusters_in_macro),
            "n_clusters": len(clusters_in_macro),
            "n_empresas": n_empresas,
            "pct_total": round(n_empresas / len(df_in) * 100, 2)
        })
    composition_df = pd.DataFrame(composition_rows)
    
    # -------- 7) Estadísticos por MACRO-CLUSTER (numéricos) + robust-z ----------
    global_median_map = dict(zip(global_num_df["variable"], global_num_df["median"])) if not global_num_df.empty else {}
    global_mad_map = dict(zip(global_num_df["variable"], global_num_df["MAD_raw"])) if not global_num_df.empty else {}
    
    macro_num_rows = []
    for macro_id, g in df_in.groupby(macro_col, sort=True):
        for col in num_cols_final:
            s = pd.to_numeric(g[col], errors="coerce")
            n = int(s.notna().sum())
            med = float(np.nanmedian(s)) if n > 0 else np.nan
            row = {
                "macro_cluster": int(macro_id),
                "variable": col,
                "n": n,
                "median": med,
                "IQR": float(iqr(s)) if n > 0 else np.nan,
                "mean": float(np.nanmean(s)) if n > 0 else np.nan,
                "std": float(np.nanstd(s, ddof=1)) if n > 1 else np.nan,
                "min": float(np.nanmin(s)) if n > 0 else np.nan,
                "max": float(np.nanmax(s)) if n > 0 else np.nan,
            }
            g_median = global_median_map.get(col, np.nan)
            g_mad = global_mad_map.get(col, np.nan)
            denom = (g_mad if pd.notna(g_mad) and g_mad > 0 else 1e-9)
            row["robust_z_median"] = float((med - g_median) / denom) if pd.notna(med) and pd.notna(g_median) else np.nan
            macro_num_rows.append(row)
    
    macro_overview_df = pd.DataFrame(macro_num_rows).sort_values(["macro_cluster", "variable"]) if macro_num_rows else pd.DataFrame()
    
    # -------- 8) Categóricas por macro-cluster ----------
    macro_cat_rows = []
    for macro_id, g in df_in.groupby(macro_col, sort=True):
        for col in cat_cols_final:
            s = g[col].astype("category")
            vc = s.value_counts(dropna=False)
            total = int(vc.sum())
            top = vc.head(top_k_modes)
            for cat_val, cnt in top.items():
                macro_cat_rows.append({
                    "macro_cluster": int(macro_id),
                    "variable": col,
                    "category": str(cat_val),
                    "count": int(cnt),
                    "pct": float(cnt / total * 100.0) if total > 0 else np.nan,
                })
    
    macro_cat_df = pd.DataFrame(macro_cat_rows).sort_values(["macro_cluster", "variable", "count"], ascending=[True, True, False]) if macro_cat_rows else pd.DataFrame()
    
    # -------- 9) Exportar a Excel ----------
    output_xlsx = Path(output_xlsx)
    output_xlsx.parent.mkdir(parents=True, exist_ok=True)
    
    with pd.ExcelWriter(output_xlsx, engine="xlsxwriter") as writer:
        # Hoja 1: Composición de macro-clusters
        composition_df.to_excel(writer, sheet_name="Composicion_MacroClusters", index=False)
        ws0 = writer.sheets["Composicion_MacroClusters"]
        ws0.freeze_panes(1, 0)
        
        # Hoja 2: Global Summary
        if not global_num_df.empty:
            startrow = 0
            global_num_df.to_excel(writer, sheet_name="Global_Summary", index=False, startrow=startrow)
            ws = writer.sheets["Global_Summary"]
            
            if not global_cat_df.empty:
                startrow = len(global_num_df) + 3
                ws.write(startrow, 0, f"Global categorical top-{top_k_modes} per variable")
                global_cat_df.to_excel(writer, sheet_name="Global_Summary", index=False, startrow=startrow+1)
        
        # Hoja 3: MacroClusters Overview (numéricas)
        if not macro_overview_df.empty:
            macro_overview_df.to_excel(writer, sheet_name="MacroClusters_Overview", index=False)
            ws2 = writer.sheets["MacroClusters_Overview"]
            ws2.freeze_panes(1, 0)
        
        # Hoja 4: Categóricas por macro-cluster
        if not macro_cat_df.empty:
            macro_cat_df.to_excel(writer, sheet_name="Categorical_Overview", index=False)
            ws3 = writer.sheets["Categorical_Overview"]
            ws3.freeze_panes(1, 0)
    
    log_info(f"Perfiles de MACRO-CLUSTERS exportados a: {output_xlsx}")
    # Convertir a int nativo para evitar np.int64 en el log
    macro_ids = [int(x) for x in sorted(df_in[macro_col].unique())]
    log_info(f"Macro-clusters encontrados: {macro_ids}")
    
    return output_xlsx

### **Pipeline provisional**

In [65]:
def provisional_pipeline(
    raw_file: str = RAW_FILE,
    sep: Optional[str] = SEP,
    run_optuna: bool = False,
    run_stability: bool = False,
    best_params_path: Optional[Path] = None,
) -> pd.DataFrame:
    """
    Provisional pipeline to test data processing steps.

    Args:
        raw_file: Path to the raw CSV file.
        sep: Separador para el CSV file (or None for auto-detection).

    Returns:
        A pandas DataFrame with processed data.
    """
    log_file = setup_logging()
    log_info(f"Iniciando pipeline provisional. Log file: {log_file}")
    ensure_dirs() # Ensure output directories exist

    # 1. Limpieza del archivo CSV
    cleaned_file = DIR_CLEAN / "afiliados_mas_usos_cleaned.csv"
    clean_csv_file(raw_file, cleaned_file)

    # 2. Cargar y tipear datos limpios
    df = cargar_y_tipear(cleaned_file, sep=';') # Use ';' as separator for the cleaned file
    log_info(f"Datos cargados y tipeados. Filas: {df.shape[0]}, Columnas: {df.shape[1]}")

    # 3. Resumen de análisis estadísitico de las variables (sin transformar)
    save_full_summary_to_excel(df, ["SeccionEconomicaEmp"], DIR_ANALYSIS) #Guardar análisis estadístico de variables

    # 4. asegurar índices
    df = ensure_row_id(df, id_col="RowID")   # crea/asegura un ID estable (no cambia)

    # 4.5) Foto de columnas originales (antes de añadir ILR / Log1p / etc.)
    original_columns = df.columns.tolist()   # <- incluye 'RowID' y todas las columnas originales

    df = df.copy()

    # 5. Aplicar ILR a los bloques composicionales
    for block_name, block_cols in BLOCKS.items():
      ilr_df = compute_block_ilr(df, block_cols, block_name)
      if not ilr_df.empty:
        # Une por índice (más claro que concat axis=1) y preserva el orden/índice original
        df = df.join(ilr_df, how="left")

    # 6. Aplicar transformación log1p(log(1+x)) a las columnas TotalAfiliados, Aporte y TotalUsos
    df_transformed = apply_log1p_transformations(df.copy())

    # 7. Preparar DataFrame que se usará para clusterización (ILR, Log1p, and original categorical for FAMD)
    df_famd_input = preparar_df_cluster(df_transformed)
    log_info("Dataframe de entrada para FAMD preparado.")


    # 8. Análisis de correlación de variables (ILR, Log1p) ANTES de FAMD y escalado
    analyze_correlation(df_famd_input, DIR_ANALYSIS) # This would fail as df_famd_input is not scaled numeric


    # 9. Guardar CSV con datos preparados para FAMD (ILR, Log1p, SeccionEconomicaEmp)
    famd_input_output_path = DIR_CLEAN / "datos_clustering_famd_input.csv"
    save_dataframe_to_csv(df_famd_input, famd_input_output_path)


    # 10. Reducir dimensionalidad con FAMD
    reduced_df_famd = reduce_dimensionality_famd(df_famd_input, "SeccionEconomicaEmp", n_components=9, random_state=RANDOM_STATE)

    # Check if FAMD reduction was successful and returned data
    if reduced_df_famd.empty:
        log_warning("La reducción FAMD devolvió un DataFrame vacío. No se puede proceder.")
        return pd.DataFrame() # Return empty dataframe if FAMD failed or returned empty


    # 11. Escalar los datos REDUCIDOS con RobustScaler
    cluster_df_scaled = scale_data(reduced_df_famd)

    # Check if scaling resulted in an empty DataFrame
    if cluster_df_scaled.empty:
        log_warning("El DataFrame escalado (después de FAMD) está vacío. No se puede proceder con el análisis de correlación ni el muestreo.")
        return cluster_df_scaled # Return empty


    # 12. Guardar CSV con datos escalados (los componentes FAMD escalados)
    scaled_output_path = DIR_CLEAN / "datos_clustering_famd_scaled.csv"
    save_dataframe_to_csv(cluster_df_scaled, scaled_output_path, index=True)  # <- importante


    # 13. Análisis de correlación de variables finales (los componentes FAMD escalados)
    #analyze_correlation(cluster_df_scaled, DIR_ANALYSIS)


    # 14. Tomar una muestra aleatoria del DataFrame escalado (componentes FAMD)
    cluster_df_sampled_scaled = sample_data(cluster_df_scaled, SAMPLE_N, RANDOM_STATE)
    log_info("DataFrame final para clusterización (componentes FAMD escalados y muestreado):")
    display(cluster_df_sampled_scaled.head())

    # 14.1 Asegurar directorio de Optuna
    optuna_db_dir = DIR_ANALYSIS / 'HiperParametros_Maestro_ajustpersist'
    optuna_db_dir.mkdir(parents=True, exist_ok=True)
    log_info(f"Directorio para la base de datos de Optuna asegurado: {optuna_db_dir}")

    # Ruta por defecto al JSON con los mejores parámetros
    default_best_params_path = optuna_db_dir / "best_hdbscan_Maestro_ajustpersist.json"
    if best_params_path is None:
        best_params_path = default_best_params_path

    # 14.2 Obtener mejores hiperparámetros: o se recalculan, o se cargan de JSON
    if run_optuna:
        log_info("[Optuna] Ejecutando tuning de hiperparámetros...")
        best_params, study = tune_hdbscan_optuna(
            X=cluster_df_sampled_scaled.values
              if isinstance(cluster_df_sampled_scaled, pd.DataFrame)
              else cluster_df_sampled_scaled,
            n_trials=400,
            study_name="hdbscan_tpe_comfama_Maestro_ajustpersist",
            storage=f"sqlite:///{optuna_db_dir / 'hdbscan_tuning_Maestro_ajustpersist.db'}",
            save_best_to=str(best_params_path.resolve()),
            target_min_mean_prob=0.80,
            target_min_persist=0.30,
            target_max_noise=20.0,
            target_min_clusters=5,
            target_max_clusters=100,
        )
    else:
        # Cargar parámetros ya ajustados
        if best_params_path.exists():
            with best_params_path.open("r", encoding="utf-8") as f:
                best_params = json.load(f)
            log_info(f"[Optuna] Usando hiperparámetros cargados de: {best_params_path}")
        
        #else:
            #raise FileNotFoundError(
            #    f"No existe el archivo con mejores hiperparámetros: {best_params_path}. "
            #    "Ejecuta la pipeline con run_optuna=True al menos una vez."
            #)


    # 15. Aplicar método de clusterización (HDBSCAN con mejores hiperparámetros)
    labels_raw, model, metrics_results = apply_hdbscan_clustering(
        cluster_df_sampled_scaled,
        min_cluster_size= best_params["min_cluster_size"], #198
        min_samples= best_params["min_samples"], #19
        cluster_selection_epsilon= best_params["cluster_selection_epsilon"], #0.4
        metric= best_params.get("metric", "euclidean"), # <- usa métrica hallada
        cluster_selection_method="eom" # <- consistente con el tuning
    )

    # 15.1 Reasignar ruido a su centroide más cercano
    labels_filled, noise_conf, was_noise = reassign_noise_to_nearest_centroid(
        X=cluster_df_sampled_scaled,
        labels=labels_raw,
        min_cluster_size_for_centroid=5,
        metric= best_params.get("metric", "euclidean")
    )

    log_info(
        f"[Noise-fill] Puntos de ruido originales: {was_noise.sum()} "
        f"({was_noise.sum() / len(was_noise) * 100:.2f}%)"
    )

    if run_stability:
        #15.2 Evaluar estabilidad con ARI (jitter + multiresolution)
        stability_dir = DIR_ANALYSIS / "Estabilidad_HDBSCAN_Maestro_ajustpersist"
        stability_jitter, stability_multires_df = run_hdbscan_stability_checks(
            X=cluster_df_sampled_scaled,
            best_params=best_params,
            output_dir=stability_dir,
            consider_noise_in_mean_prob=False,
            random_state=RANDOM_STATE,
        )

        log_info(
            "[Stability] Jitter -> "
            f"mean_ari={stability_jitter['mean_ari']:.3f}, "
            f"min_ari={stability_jitter['min_ari']:.3f}, "
            f"max_ari={stability_jitter['max_ari']:.3f}, "
            f"mean_n_clusters={stability_jitter['mean_n_clusters']:.1f}, "
            f"mean_noise_pct={stability_jitter['mean_noise_pct']:.2f}%"
        )
    
    # 15.3 Crear macro-clusters (usando labels_raw sin ruido reasignado)
    macro_cluster_map = create_macro_clusters(
        X=cluster_df_sampled_scaled,
        labels_raw=labels_raw,
        max_macro_clusters=8,  # Ajusta según tu preferencia
        linkage_method="ward",
        min_cluster_size_for_centroid=5
    )
    
    log_info(f"Mapeo cluster -> macro_cluster: {macro_cluster_map}")

    # 16. Retornar csv original pero ya clasificado + macro-clusters
    clustered_csv_path = DIR_MODELS / "afiliados_mas_usos_clusterizados.csv"
    write_clustered_full_csv_with_macro(
        original_df=df,
        labels_raw=labels_raw,
        labels_filled=labels_filled,
        was_noise=was_noise,
        noise_confidence=noise_conf,
        sampled_index=cluster_df_sampled_scaled.index,
        macro_cluster_map=macro_cluster_map,  # <--PARÁMETRO
        out_path=clustered_csv_path,
        id_col="RowID",
        original_columns=original_columns
    )


    # 17. Crear perfiles por cluster - Descripción de cada grupo
    df_clustered = pd.read_csv(clustered_csv_path)
    profiles_xlsx_path = DIR_MODELS / "perfil_clusters.xlsx"
    generate_cluster_profiles_excel(
        labeled_df=df_clustered,
        output_xlsx=profiles_xlsx_path,
        id_col="RowID",
        cluster_col="cluster_label",
        exclude_noise=True, # Al poner False estamos teniendo en cuenta los datos que eran ruido, al realizar los perfiles
        exclude_unseen=True,
        categorical_cols=["DocEmp","TipoDocEmp","NomEmp","CodclaseEconomicaEmp","CodSeccionEconomicaEmp","SeccionEconomicaEmp","NitSinDV"], # Pasar lista explicita
        top_k_modes=5
    )

    # 18. Crear perfiles de MACRO-CLUSTERS - Descripción de cada macro-grupo
    macro_profiles_xlsx_path = DIR_MODELS / "perfil_macro_clusters.xlsx"
    generate_macro_cluster_profiles_excel(
        labeled_df=df_clustered,
        output_xlsx=macro_profiles_xlsx_path,
        id_col="RowID",
        macro_col="macro_cluster",
        cluster_col="cluster_label",
        exclude_noise=True,
        exclude_unseen=True,
        categorical_cols=["DocEmp","TipoDocEmp","NomEmp","CodclaseEconomicaEmp","CodSeccionEconomicaEmp","SeccionEconomicaEmp","NitSinDV"],
        top_k_modes=5
    )

    '''
    #Por ahora comentamos para no saturar el notebook
    # 19. Reducción de dimensionalidad (para visualización) - Apply TSNE on the data used for clustering
    reduced_df_tsne = reduce_dimensionality_tsne(cluster_df_sampled_scaled, n_components=3)
    log_info("DataFrame reducido a 3 dimensiones con TSNE para visualización de clusters (sobre componentes FAMD).")

    # 20. Gráfico de clusters en 3D
    plot_clusters_tsne(reduced_df_tsne, labels)

    #return df_transformed
    '''


In [66]:
# Execute the provisional pipeline
provisional_pipeline(run_optuna=False, run_stability=True)

2025-12-11 14:30:28,765 - INFO - Iniciando pipeline provisional. Log file: outputs\pipeline_log_20251211_143028.log
2025-12-11 14:30:28,770 - INFO - Detectando separador para: afiliados_mas_usos.csv
2025-12-11 14:30:28,786 - INFO - Separador detectado: ','
2025-12-11 14:30:29,680 - INFO - Headers detectados: True (coincidencias=34)
2025-12-11 14:30:31,676 - INFO - CSV limpio → outputs\01_clean\afiliados_mas_usos_cleaned.csv
2025-12-11 14:30:31,986 - INFO - Columnas del DataFrame después de cargar y tipear: ['DocEmp', 'TipoDocEmp', 'NomEmp', 'CodclaseEconomicaEmp', 'CodSeccionEconomicaEmp', 'SeccionEconomicaEmp', 'NitSinDV', 'Femenino', 'Masculino', 'Desconocido', 'TarifaA', 'TarifaB', 'TarifaC', 'Jovenes', 'AdultoJoven', 'Adulto', 'AdultoMayor', 'TotalAfiliados', 'Aporte', 'TotalUsos', 'UsosCultura', 'UsosEducacion', 'UsosSalud', 'UsosEmpleo', 'PorcentajeFemenino', 'PorcentajeMasculino', 'PorcentajeDesconocido', 'PorcentajeTarifaA', 'PorcentajeTarifaB', 'PorcentajeTarifaC', 'Porcentaje

,GENERO_ILR_1,GENERO_ILR_2,TARIFA_ILR_1,TARIFA_ILR_2,EDAD_ILR_1,EDAD_ILR_2,EDAD_ILR_3,Log1p_TotalAfiliados,Log1p_Aporte,Log1p_IntensidadUsos,SeccionEconomicaEmp
RowID,,,,,,,,,,,
0,-0.234045,10.703233,2.229026,9.959205,-0.058329,-0.043198,11.012403,3.044522,14.056082,0.053541,S_Otras Actividades de serv
1,0.776836,10.596918,0.000000,-0.565952,-9.278911,5.357182,-7.576199,1.609438,13.925995,0.308301,G_Comercio y Rep. Automotores
2,1.786724,10.185978,2.602514,1.502562,-0.611793,1.127602,1.415475,3.751463,15.199386,0.129515,G_Comercio y Rep. Automotores
3,0.292605,10.697011,9.769040,5.640158,-0.804506,-0.292815,1.417603,3.286534,14.370075,0.077800,"I_Alojamiento, serv. de comida"
4,-0.286707,10.697699,9.769040,5.640158,-0.294652,-1.141211,0.606218,3.349904,14.379351,0.062643,F_Construcción


2025-12-11 14:30:48,667 - INFO - Dataframe de entrada para FAMD preparado.
2025-12-11 14:30:48,667 - INFO - Iniciando análisis de correlación.
2025-12-11 14:30:48,691 - INFO - Matriz de correlación calculada.
2025-12-11 14:30:48,693 - INFO - Generando Pairplot (puede tardar)...
2025-12-11 14:31:09,876 - INFO - Pairplot guardado en: outputs\02_analysis\correlation_pairplot_pre_famd.png
2025-12-11 14:31:09,877 - INFO - Generando Heatmap de correlación.
2025-12-11 14:31:10,179 - INFO - Heatmap guardado en: outputs\02_analysis\correlation_heatmap_pre_famd.png
2025-12-11 14:31:10,179 - INFO - Análisis de correlación completado.
2025-12-11 14:31:10,788 - INFO - CSV guardado en: outputs\01_clean\datos_clustering_famd_input.csv (index=True)
2025-12-11 14:31:10,788 - INFO - Iniciando reducción FAMD a 9 componentes, incluyendo 'SeccionEconomicaEmp'.
2025-12-11 14:31:11,229 - INFO - Varianza explicada por los primeros 9 componentes FAMD (calculada de autovalores): 0.9089 (90.89%)
2025-12-11 14:31

,eigenvalue,% of variance,% of variance (cumulative)
component,,,
0,2.367,12.14%,12.14%
1,1.453,7.45%,19.59%
2,1.350,6.93%,26.51%
3,0.973,4.99%,31.50%
4,0.911,4.67%,36.17%
5,0.823,4.22%,40.39%
6,0.775,3.97%,44.37%
7,0.735,3.77%,48.14%
8,0.602,3.09%,51.22%


2025-12-11 14:31:13,356 - INFO - Reducción FAMD completada (9 componentes retenidos).


,FAMD1,FAMD2,FAMD3,FAMD4,FAMD5,FAMD6,FAMD7,FAMD8,FAMD9
RowID,,,,,,,,,
0,1.314651,-0.963388,0.045222,-1.417771,-0.620825,0.485464,-0.168141,-1.348798,0.236551
1,0.787059,1.873269,0.209500,0.583979,-0.135204,-0.043957,-0.656358,2.616962,0.377539
2,2.434970,0.129700,-0.177579,0.084368,0.620457,0.469479,-0.167763,0.398059,-0.123240
3,1.139830,-0.546513,-0.631428,-0.907355,1.307110,-0.316533,-0.155691,0.293540,-0.630705
4,1.255824,-0.361623,-1.406009,-0.681282,0.793991,0.081230,0.218098,0.572515,-0.529140


2025-12-11 14:31:13,365 - INFO - Iniciando escalado con RobustScaler para todas las columnas numéricas.
2025-12-11 14:31:13,393 - INFO - Escalado completado con RobustScaler. DataFrame final: 54026 filas, 9 columnas.


,FAMD1,FAMD2,FAMD3,FAMD4,FAMD5,FAMD6,FAMD7,FAMD8,FAMD9
RowID,,,,,,,,,
0,0.758663,-0.511976,-0.027322,-1.184607,-0.578155,0.506962,-0.159417,-0.989606,0.349484
1,0.492387,1.082625,0.089811,0.595176,-0.180979,0.051952,-0.756158,2.349213,0.505531
2,1.324091,0.102494,-0.186183,0.150965,0.437055,0.493224,-0.158954,0.481094,-0.048737
3,0.670431,-0.277633,-0.509785,-0.730789,0.998650,-0.182313,-0.144199,0.393098,-0.610405
4,0.728973,-0.173699,-1.062075,-0.529784,0.578984,0.159543,0.312680,0.627970,-0.497992


2025-12-11 14:31:14,646 - INFO - CSV guardado en: outputs\01_clean\datos_clustering_famd_scaled.csv (index=True)
2025-12-11 14:31:14,648 - INFO - No se ha especificado un tamaño de muestra o el tamaño es mayor/igual al total de filas. Usando todos los datos.
2025-12-11 14:31:14,651 - INFO - DataFrame final para clusterización (componentes FAMD escalados y muestreado):


,FAMD1,FAMD2,FAMD3,FAMD4,FAMD5,FAMD6,FAMD7,FAMD8,FAMD9
RowID,,,,,,,,,
0,0.758663,-0.511976,-0.027322,-1.184607,-0.578155,0.506962,-0.159417,-0.989606,0.349484
1,0.492387,1.082625,0.089811,0.595176,-0.180979,0.051952,-0.756158,2.349213,0.505531
2,1.324091,0.102494,-0.186183,0.150965,0.437055,0.493224,-0.158954,0.481094,-0.048737
3,0.670431,-0.277633,-0.509785,-0.730789,0.998650,-0.182313,-0.144199,0.393098,-0.610405
4,0.728973,-0.173699,-1.062075,-0.529784,0.578984,0.159543,0.312680,0.627970,-0.497992


2025-12-11 14:31:14,656 - INFO - Directorio para la base de datos de Optuna asegurado: outputs\02_analysis\HiperParametros_Maestro_ajustpersist
2025-12-11 14:31:14,664 - INFO - [Optuna] Usando hiperparámetros cargados de: outputs\02_analysis\HiperParametros_Maestro_ajustpersist\best_hdbscan_Maestro_ajustpersist.json
2025-12-11 14:31:14,664 - INFO - HDBSCAN(metric=euclidean, min_cluster_size=198, min_samples=19, cluster_selection_method=eom, cluster_selection_epsilon=0.39980276970238043, allow_single_cluster=False)
2025-12-11 14:31:40,911 - INFO - HDBSCAN: clusters=37 (sin contar ruido), ruido=19.86%
2025-12-11 14:32:00,912 - INFO - Métricas HDBSCAN -> n_clusters=37, ruido=19.86%, silhouette=0.30413699385251697, CH=3651.092933212947, DB=1.1897535755354507, mean_prob=0.8585588289296227, median_persist_raw=0.11063602283798417, median_persist_main_w=0.14843445590800997
2025-12-11 14:32:00,933 - INFO - [Noise-fill] Puntos de ruido originales: 10732 (19.86%)
2025-12-11 14:32:00,942 - INFO - 